<div dir="rtl">

# رافِدَين ٢ — دفتر RunPod الشامل لبناء أقوى نموذج استنساخ لحظي للهجة العراقية

**رحلة كاملة في دفتر واحد:** تنزيل بيانات عراقية حقيقية ← تجهيز وتقطيع ← تدريب ضبط دقيق على النموذج الرسمي Seed-VC (tiny-XLSR) ← **تقييم موضوعي A/B يثبت التحسين بالأرقام** ← قياس أداء الاستنساخ اللحظي ← حفظ دائم متعدد الطبقات (قرص شبكي + HuggingFace + أرشيف جاهز).

النموذج الناتج يعمل فوراً مع خط الاستدلال الرسمي لـ Seed-VC ومع خدمة «رافِدَين» الإنتاجية (REST + WebSocket للبث اللحظي في المكالمات).

---

## ما الجديد في هذا الإصدار الثاني؟

| # | الإضافة | لماذا تصنع فرقاً؟ |
|---|---|---|
| 1 | **خلية تقييم A/B ذهبية** | تحوّل النموذج الأساس والنموذج المضبوط على **نفس أزواج الاختبار العراقية** بنفس البذرة العشوائية، ويقيس موضوعياً: تشابه المتحدث (CAMPPlus) + الحفاظ على المحتوى (XLSR). أرقام لا انطباعات |
| 2 | **بيانات اختبار معزولة (Held-out)** | تُستبعد عشرات المقاطع العراقية من التدريب قبل انطلاقه، فيقيَّم النموذج على كلام **لم يره إطلاقاً** — الطريقة العلمية الوحيدة لإثبات التحسن |
| 3 | **بنشمارك الاستنساخ اللحظي الحقيقي** | يحاكي بروفايل الإنتاج `balanced` (كتل 180ms، 10 خطوات تدفق، crossfade 40ms) بنفس منطق محرك رافِدَين الإنتاجي، ويعطيك p50/p95 وزمن الوصول الشامل على كرتك الفعلي |
| 4 | **تقرير بيانات بالرسم البياني** | مدرّج توزيع أطوال المقاطع + ساعات كل مصدر — تفحص جودة بيانات التدريب قبل حرق ساعات GPU |
| 5 | **حارس فقد NaN** | المراقبة تكشف فوراً انهيار دالة الخسارة (NaN/Inf) بدل اكتشافه بعد ساعات |
| 6 | **توثيق قابلية إعادة الإنتاج** | بصمات git لنسختَي seed-vc وrafidain + كل إعدادات التشغيل تُحفظ داخل حزمة النموذج نفسها |
| 7 | **حراس صلابة إضافيون** | تحقق وجود GPU قبل التدريب، تحقق اكتمال النقاط الأساسية، فحص مساحة القرص قبل التنزيل الكبير، وإصلاح تلقائي لتسريب بيانات الاختبار |

---

## ضمانات صادقة — اقرأها مرة واحدة

**ما يضمنه هذا الدفتر حرفياً:**
- كل أمر تدريب يستدعي `train.py` **الرسمي** لمستودع Seed-VC كما هو، بنفس نقطة البداية المنشورة `DiT_uvit_tat_xlsr_ema.pth` ونفس معدل التعلم المعتمد upstream (1e-5) — لا كود تدريب مبتكر قد يكسر التوليف.
- كل مرحلة **قابلة للاستئناف**: انقطع الاتصال أو أُعيد تشغيل النواة أو أُوقفت الحاوية؟ أعد تشغيل الخلايا بالترتيب وتكمل من حيث توقفت.
- التدريب يعمل **كعملية خلفية مستقلة** عن النواة والمتصفح.
- **ثلاث طبقات حفظ دائم**: قرص RunPod الشبكي (يبقى بعد إيقاف/حذف الحاوية)، مستودع HuggingFace خاص بك (مجاني، للأبد)، وأرشيف `tar.gz` واحد جاهز للتنزيل.
- التقييم A/B **يقيس** التحسن بالأرقام قبل أن تعتمد النموذج — ولن تجد نفسك تتخيل تحسناً غير موجود.

**ما لا يستطيع أي كود على الأرض ضمانه (ومن يدّعي غير ذلك يبيعك وهماً):**
- الدرجة النهائية لجودة اللهجة تعتمد على: كمية بيانات العراقي النظيفة المتاحة، وعدد الخطوات، وقوة GPU. لهذا جهّزنا الدفتر ليجلب **حتى 339 ساعة** عراقية موثّفة الجودة، ويقيس النتيجة بدل تخمينها.
- القاعدة العملية المجرّبة: 200 ساعة × 3000 خطوة على RTX 4090 تعطي فتحاً واضحاً في طبيعية اللهجة؛ ويمكنك دائماً مواصلة التدريب من آخر نقطة أو رفع `STEPS`.

> **الخلاصة:** الكود مضمون الصحة والمسار مضمون الاستئناف والقياس مضمون الموضوعية — والنموذج يزداد قوة كلما أعطيته بيانات وخطوات أكثر، والدليل أمامك بالأرقام في نهاية الدفتر.

---

## ما تحتاجه قبل البدء

1. حساب RunPod فيه رصيد (تقديرات الكلفة أدناه).
2. (اختياري لكن موصى به بشدة) حساب HuggingFace مجاني + رمز بصلاحية Write — للحفظ الدائم في الطبقة الثانية.

| السيناريو | GPU مقترح | زمن تقريبي | كلفة تقريبية |
|---|---|---|---|
| تجربة سريعة (25 ساعة × 1500 خطوة) | RTX 4090 | ~2–3 ساعات | ~1–2 دولار |
| **الموصى به** (200 ساعة × 3000 خطوة) | RTX 4090 | ~4–6 ساعات | ~2–4 دولارات |
| أقصى جودة (339 ساعة × 4000+ خطوة) | A40 / A100 | ~6–10 ساعات | ~5–12 دولاراً |

</div>


<div dir="rtl">

## ١) تجهيز الحاوية على RunPod (قبل فتح الدفتر)

### أ. (موصى به) قرص شبكي دائم — Network Volume

كل ما يبقى على قرص الحاوية المحلي يُفقد عند حذف الحاوية (Terminate). القرص الشبكي يبقى **للأبد** مهما فعلت بالحاويات:

1. من لوحة [runpod.io](https://runpod.io) → **Storage** → **Network Volume** → **+ New**.
2. الحجم: **100GB** يكفي لـ 200 ساعة (بحجم أكبر لـ «الكتالوج كله» + بياناتك الخاصة).
3. أنشئه في نفس المنطقة (Region) التي ستختار فيها الـ GPU لتقليل الكلفة والزمن.

### ب. إنشاء حاوية GPU

1. **GPU Pods** → **Deploy** على أحد هذه الخيارات (كلها مجرّبة مع هذا الدفتر):

| GPU | الذاكرة | الدفعة التلقائية | ملاحظات |
|---|---|---|---|
| **RTX 4090** | 24GB | 4 | **الأفضل للسعر/الأداء — موصى به** |
| RTX 3090 | 24GB | 4 | أرخص قليلاً وأبطأ قليلاً |
| A40 / L40S | 48GB | 6 | ممتاز للجلسات الطويلة |
| A100 80GB | 80GB | 8 | أسرع وأغلى — للكتالوج الكامل |
| T4 | 16GB | 2 | يعمل لكن بطيء — للتجارب فقط |

2. القالب (Template): اختر **RunPod Pytorch 2.4.0** (أو أي قالب يحتوي JupyterLab — الدفتر يبني بيئته المعزولة بنفسه ولا يعتمد على قالب محدد).
3. الإعدادات:
   - **Volume Disk**: وصّل القرص الشبكي الذي أنشأته على المسار `/workspace`.
   - **Container Disk**: **30GB** على الأقل (تنزيل PyTorch + النقاط + مخابئ مؤقتة).
4. اضغط **Deploy** ثم **Connect** → **Connect to Jupyter Lab**.
5. ارفع هذا الدفتر (زر الرفع في أعلى اللوحة اليسرى) وافتحه.

### ج. لماذا هذا الترتيب مهم؟

| القرار | الأثر |
|---|---|
| قرص شبكي على `/workspace` | بياناتك ونقاط التدريب والحزم تبقى بعد أي إيقاف أو حذف |
| بيئة Python معزولة على القرص الشبكي | إعادة تشغيل الحاوية لا تعيد تثبيت شيء — تستأنف خلال ثوانٍ |
| تثبيت PyTorch المثبّت (2.4.0 cu121) | نفس الإصدار الذي تتحقق منه سكربتات المستودع الرسمية — لا مفاجآت توافق |

</div>


<div dir="rtl">

## ٢) خريطة الدفتر — شغّل بالترتيب من الأعلى

| # | الخلية | الوصف | المدة التقريبية |
|---|---|---|---|
| 1 | الإعدادات المركزية | كل مقابض التحكم + حفظها على القرص الدائم | ثوانٍ |
| 2 | فحص البيئة | GPU / قرص / أدوات + تحديد الدفعة تلقائياً | ثوانٍ |
| 3 | تبعيات النظام + uv | ffmpeg / git / مدير حزم سريع | ~دقيقة |
| 4 | بيئة Python المعزولة | venv دائم + PyTorch 2.4.0 cu121 + المكتبات | 3–8 دقائق (مرة واحدة) |
| 5 | استنساخ المستودعين | Seed-VC (المدرب الرسمي) + رافِدَين (خط البيانات) + بصمات git | ~دقيقة |
| 6 | استخراج خط بيانات العراق | الكتالوج المدمج: 49,194 مقطعاً / 339 ساعة | ثوانٍ |
| 7 | المجموعات الصغيرة النظيفة | youssef / shiekh / safaa من HuggingFace | 5–15 دقيقة |
| 8 | بيانات YODAS العراقية | الجلب الكبير (الساعات التي حددتها) | 15–60 دقيقة |
| 9 | (اختيارية) بياناتك الخاصة | ارفع تسجيلاتك وادمجها | حسبك |
| 10 | التقطيع والترميز | كل المقاطع ← Opus 24kHz بمقاطع 1–29.5 ث | 10–40 دقيقة |
| 11 | **تقرير البيانات** | ساعات كل مصدر + مدرّج الأطوال | ثوانٍ |
| 12 | **عزل بيانات الاختبار** | استبعاد مقاطع من التدريب للتقييم الأمين | ثوانٍ |
| 13 | تنزيل نقاط النموذج | الأساس + XLSR + HiFT + CAMPPlus (~2.2GB) | 5–15 دقيقة |
| 14 | إطلاق التدريب | عملية خلفية مستقلة عن النواة والمتصفح | — |
| 15 | مراقبة التدريب | شريط تقدم + منحنى خسارة + حارس NaN | عند الرغبة |
| 16 | (اختيارية) انتظار الاكتمال | مراقبة تلقائية حتى النهاية | — |
| 17 | تحقق أولي (دخان) | استدلال حقيقي بالنقطة المضبوطة | ~2 دقيقة |
| 18 | **التقييم A/B الذهبي** | الأساس مقابل المضبوط بالأرقام + مقاطع صوتية | ~5 دقائق |
| 19 | **بنشمارك الاستنساخ اللحظي** | RTF + زمن الوصول الشامل على بروفايل الإنتاج | ~3 دقائق |
| 20 | تصدير الحزمة النهائية | أوزان + إعداد + تقارير + بصمات | ثوانٍ |
| 21 | أرشيف tar.gz | ملف واحد جاهز للتنزيل | ثوانٍ |
| 22 | الرفع الدائم إلى HuggingFace | مستودع خاص في حسابك — للأبد | 1–3 دقائق |
| 23 | (اختيارية) rclone سحابي | Google Drive / S3 / … | — |
| 24 | (اختيارية) تحرير المساحة | حذف المصادر الخام القابلة لإعادة الجلب | — |
| 25 | دليل النشر | كيف تستخدم النموذج في الإنتاج | للقراءة |

**قواعد ذهبية:**
- كل خلية **آمنة الإعادة**: تشغيلها مرة أخرى لا يكسر شيئاً ولا يكرر العمل المنجز.
- إذا انقطع كل شيء وأعدت تشغيل الحاوية والنواة: شغّل الخلايا **بالترتيب من 1** — كل منها يكتشف المنجز ويتخطاه.
- التدريب نفسه (الخلية 14) مستقل عن الدفتر تماماً: يمكنك إغلاق المتصفح وإعادة فتحه لاحقاً.

</div>


In [ ]:
# ============================================================ #
#  الخلية ١ — الإعدادات المركزية والأدوات المساعدة              #
#  كل مقابض التحكم في مكان واحد، وتُحفظ على القرص الدائم        #
#  (إعادة تشغيل النواة لا تفقدها — وتُعاد عند الاستئناف)         #
# ============================================================ #
import os, sys, json, subprocess, shutil, time, re
from pathlib import Path

NB_VERSION = "2.1"

# ---------- 1) جذر العمل: على تخزين دائم إن وُجد ----------
# على RunPod: القرص الشبكي (Network Volume) يُوصل تلقائياً في /workspace وهو دائم.
def _pick_root():
    # IRAQI_DATA_ROOT = مسار جذر العمل كاملاً مباشرة (نفس دلالته في سكربتات المستودع)
    override = os.environ.get("IRAQI_DATA_ROOT", "").strip()
    if override:
        return Path(override)
    for cand in ("/workspace", "/root"):
        p = Path(cand)
        try:
            p.mkdir(parents=True, exist_ok=True)
            (p / ".write_test").write_text("ok"); (p / ".write_test").unlink()
            return p / "iraqi_seedvc"
        except Exception:
            continue
    return Path.home() / "iraqi_seedvc"

ROOT = _pick_root()   # /workspace/iraqi_seedvc (دائم) أو /root/iraqi_seedvc (مؤقت)
ROOT.mkdir(parents=True, exist_ok=True)
CFG_PATH  = ROOT / "nb_config.json"

# ---------- 2) المقابض الافتراضية ----------
DEFAULTS = dict(
    # البيانات
    YODAS_HOURS    = 200,    # ساعات YODAS العراقية (0 = تعطيل، "all" = الكل 339 ساعة، أو أي رقم)
    SKIP_SMALL     = False,  # تخطي المجموعات النظيفة الخمس (youssef/shiekh/safaa/lahgtna/kharrufa)
    # التدريب
    RUN_NAME       = "iraqi_v1",  # اسم التشغيل (مجلد النقاط والحزمة)
    STEPS          = 3000,   # عدد خطوات الضبط الدقيق (500 لمتحدث واحد، 1500-4000 للهجة عموماً)
    BATCH          = "auto", # حجم الدفعة: "auto" = حسب ذاكرة GPU، أو رقم 1/2/4/6/8
    SAVE_EVERY     = 250,    # حفظ نقطة تفتيش كل N خطوة (يُحتفظ بآخر نقطتين تلقائياً)
    TRAIN_WORKERS  = 2,      # عمال تحميل بيانات التدريب
    # التنزيل والتجهيز
    FETCH_WORKERS  = 16,     # تنزيلات YODAS المتوازية (ارفعها إلى 32 إذا كانت الشبكة سريعة)
    PREP_WORKERS   = 0,      # عمال التقطيع والترميز (0 = تلقائي: حتى 16 نواة)
    HF_ENDPOINT    = "",     # مرآة HuggingFace إن لزم (مثال: https://hf-mirror.com) — اتركها فارغة عادة
    # التقييم (جديد في الإصدار ٢)
    HOLDOUT_COUNT  = 24,     # عدد مقاطع الاختبار المعزولة عن التدريب (تقييم أمين)
    EVAL_PAIRS     = 5,      # أزواج A/B في التقييم الذهبي
    EVAL_STEPS     = 25,     # خطوات التدفق أثناء التقييم (متساوية للنموذجين = مقارنة عادلة)
    # التخزين الدائم (HuggingFace Hub)
    HF_TOKEN       = "",     # رمز من hf.co/settings/tokens بصلاحية Write — للرفع الدائم (الخلية 22)
    HF_REPO_ID     = "",     # فارغ = <اسم-حسابك>/rafidain-seedvc-iraqi تلقائياً
    HF_PRIVATE     = True,   # مستودع خاص لا يراه أحد غيرك
    # تنظيف
    CLEAN_RAW_AFTER = False, # حذف المصادر الخام بعد تجهيز بيانات التدريب (يوفر مساحة — الخلية 24)
)

# ---------- 3) إعادة تحميل الإعدادات المحفوظة (استئناف بعد إعادة تشغيل النواة) ----------
CFG = dict(DEFAULTS)
FORCE_RESET = False   # اجعلها True مؤقتاً لإعادة كل الإعدادات إلى الافتراضية
if CFG_PATH.exists() and not FORCE_RESET:
    try:
        saved = json.loads(CFG_PATH.read_text(encoding="utf-8"))
        CFG.update({k: saved[k] for k in DEFAULTS if k in saved})
        print(f"[إعدادات] عيد تحميلها من {CFG_PATH} (عدّل ثم أعد التشغيل، أو FORCE_RESET=True للبدء من جديد)")
    except Exception as e:
        print(f"[إعدادات] تعذر قراءة المحفوظة ({e}) — نستخدم الافتراضية")
# ← عدّل أي قيمة هنا ثم أعد تشغيل الخلية، مثال:
#    CFG["YODAS_HOURS"] = 50
#    CFG["STEPS"] = 1000
CFG["NB_VERSION"] = NB_VERSION
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---------- 4) المسارات الثابتة (كلها تحت الجذر الدائم) ----------
SEEDVC   = ROOT / "seed-vc"                    # مستودع Seed-VC (المدرب الأصلي)
RAFIDAIN = ROOT / "rafidain"                   # مستودع رافِدَين (الخدمة + خط البيانات)
PIPE     = ROOT / "iraqi_pipeline.py"          # خط بيانات العراق — يُستخرج في الخلية 6
DATA_DIR = ROOT / "data" / "iraqi_finetune"    # بيانات التدريب المجهزة (Opus 24kHz)
TEST_DIR = ROOT / "data" / "iraqi_test"        # مقاطع اختبار معزولة لا يدخلها التدريب
RUN_DIR  = SEEDVC / "runs" / CFG["RUN_NAME"]   # نقاط التدريب والسجلات
EXPORTS  = ROOT / "exports"                    # حزم النموذج النهائية
VENV_PY  = ROOT / "venv" / "bin" / "python"    # بايثون البيئة المعزولة (الخلية 4)
for p in (ROOT, EXPORTS, RUN_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ---------- 5) بيئة الأوامر وأداة تشغيل موحدة ----------
ENV = dict(os.environ)
ENV["PYTHONUNBUFFERED"] = "1"
ENV["HF_HUB_DISABLE_XET"] = "1"           # تنزيل HTTP قابل للاستئناف (نفس خيار سكربتات المستودع)
ENV["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
if CFG["HF_ENDPOINT"]:
    ENV["HF_ENDPOINT"] = CFG["HF_ENDPOINT"]

def sh(cmd, cwd=None, env=None, check=True, echo=True):
    # تشغيل أمر وطباعة مخرجاته لحظياً داخل الخلية — يعيد رمز الخروج
    cmd = [str(c) for c in cmd]
    if echo:
        print("$ " + " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None,
                            env=env or ENV, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"فشل الأمر (رمز {rc}): {' '.join(cmd)}")
    return rc

# ---------- 6) حارس بيانات الاختبار: يُستدعى قبل التدريب لمنع تسريبها ----------
def reapply_holdout(verbose=True):
    """أعد فصل مقاطع الاختبار إن عادت لمجلد التدريب (بعد إعادة تشغيل prep مثلاً)."""
    hl = ROOT / "data" / "holdout_list.json"
    if not hl.exists():
        return 0
    moved = 0
    for rel in json.loads(hl.read_text()):
        src = DATA_DIR / rel
        if src.exists():
            TEST_DIR.mkdir(parents=True, exist_ok=True)
            src.rename(TEST_DIR / src.name)
            moved += 1
    if moved and verbose:
        print(f"[اختبار] أُعيد عزل {moved} مقطعاً إلى {TEST_DIR} (منع تسريب للتدريب)")
    return moved

# ---------- 7) ملخص ----------
print("=" * 64)
print(f"الجذر الدائم للعمل : {ROOT}")
print("=" * 64)
print(f"  • ساعات YODAS        : {CFG['YODAS_HOURS']}")
print(f"  • التدريب            : {CFG['STEPS']} خطوة، حفظ كل {CFG['SAVE_EVERY']}, دفعة {CFG['BATCH']}")
print(f"  • التقييم            : {CFG['EVAL_PAIRS']} أزواج A/B على {CFG['HOLDOUT_COUNT']} مقطعاً معزولاً")
print(f"  • اسم التشغيل        : {CFG['RUN_NAME']}")
print(f"  • رمز HuggingFace    : {'مضبوط (سيُرفع للأبد في الخلية 22)' if CFG['HF_TOKEN'] else 'غير مضبوط (اختياري — الرفع الدائم)'}")
print(f"\n[المسارات]  التدريب : {DATA_DIR}")
print(f"            الاختبار: {TEST_DIR}")
print(f"            النقاط  : {RUN_DIR}")
print(f"            التصدير : {EXPORTS}")
print(f"\nملاحظة: الرمز السري (إن وضعته) يُحفظ في {CFG_PATH} على قرصك الدائم فقط.")


In [ ]:
# ============================================================ #
#  الخلية ٢ — فحص البيئة: GPU / قرص / أدوات                    #
#  (يحدد حجم الدفعة تلقائياً ويكشف مدى ديمومة التخزين)          #
# ============================================================ #
import subprocess, os, shutil, re, json
from pathlib import Path

def gpu_query():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
        return [l.strip() for l in out if l.strip()]
    except Exception:
        return []

gpus = gpu_query()
if gpus:
    for i, g in enumerate(gpus):
        parts = [p.strip() for p in g.split(",")]
        name, vram_mib, driver = parts[0], float(parts[1]), parts[2]
        print(f"[GPU {i}] {name} — ذاكرة {vram_mib/1024:.1f}GB — مشغل {driver}")
else:
    print("[GPU] لا يوجد GPU ظاهر عبر nvidia-smi — لن يعمل التدريب (تجهيز البيانات فقط).")

def auto_batch(vram_gb):
    # دفعة آمنة لنموذج tiny (25M) مع مكوناته الثقيلة (XLSR-300m + HiFT + OpenVoice)
    if vram_gb >= 70: return 8
    if vram_gb >= 40: return 6
    if vram_gb >= 20: return 4
    if vram_gb >= 15: return 2
    return 1

if gpus:
    vram_gb = float([p.strip() for p in gpus[0].split(",")][1]) / 1024
    CFG["_BATCH"] = auto_batch(vram_gb)
else:
    vram_gb = 0.0
    CFG["_BATCH"] = 2
if str(CFG["BATCH"]).lower() == "auto":
    BATCH = CFG["_BATCH"]
    print(f"[دفعة] تلقائي => BATCH = {BATCH} (حسب {vram_gb:.0f}GB ذاكرة)")
else:
    BATCH = int(CFG["BATCH"])
    print(f"[دفعة] يدوي  => BATCH = {BATCH}")
CFG["_BATCH"] = BATCH
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- المعالج والذاكرة ----
cpu = os.cpu_count() or 4
try:
    mem_kb = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text()).group(1))
    print(f"[نظام] {cpu} نواة معالج — ذاكرة RAM ≈ {mem_kb/1e6:.1f}GB")
except Exception:
    print(f"[نظام] {cpu} نواة معالج")

# ---- القرص ونوع التخزين (ديمومة الملفات!) ----
du = shutil.disk_usage(ROOT)
print(f"[قرص] المتاح على {ROOT}: حر {du.free/1e9:.1f}GB من {du.total/1e9:.1f}GB")
def find_mount(path):
    best, fstype, dev = "", "", ""
    try:
        for line in Path("/proc/mounts").read_text().splitlines():
            parts = line.split()
            if len(parts) >= 3 and str(path).startswith(parts[1]) and len(parts[1]) > len(best):
                best, fstype, dev = parts[1], parts[2], parts[0]
    except Exception:
        pass
    return best, fstype, dev
mnt, fstype, dev = find_mount(ROOT)
print(f"[تخزين] نقطة الوصل: {mnt or '/'} — النوع: {fstype or 'غير معروف'} — الجهاز: {dev or '?'}")
if str(ROOT).startswith("/workspace"):
    print("        [جيد] العمل تحت /workspace — إن كان هذا القرص الشبكي (Network Volume) فملفاتك")
    print("               دائمة وتبقى بعد إيقاف/حذف الحاوية (لا تحذف الـ Volume من لوحة RunPod).")
elif str(ROOT).startswith("/root") or str(ROOT).startswith("/home"):
    print("        [تحذير] لا يوجد /workspace — أنت على قرص الحاوية المحلي:")
    print("                يبقى عند الإيقاف (Stop) لكنه يضيع عند الحذف (Terminate).")
    print("                لضمان الديمومة: أنشئ Network Volume في RunPod وأعد تشغيل الدفتر،")
    print("                أو اضبط IRAQI_DATA_ROOT، أو اعتمد على الرفع الدائم (الخلية 22).")

# ---- تقدير مساحة التنزيل المطلوبة ----
need_gb = 2.2 + 6.0   # نقاط النماذج + بيئة Python/PyTorch
t = str(CFG["YODAS_HOURS"]).strip().lower()
hours_label = "0"
if t not in ("0", ""):
    hours_label = "كل الكتالوج" if t in ("all", "كل") else f"{float(t):.0f} ساعة"
    # الخام ~ 3GB لكل 20 ساعة تقريباً (WebM) + المقاطع المشفرة Opus ~ 0.15GB/ساعة
    est_raw = 3.0 * (339 if t in ("all", "كل") else float(t)) / 20.0
    need_gb += est_raw + est_raw * 0.15
print(f"[تقدير] يحتاج هذا التشغيل ({hours_label} يوداس + مجموعات صغيرة) ≈ {need_gb:.0f}GB تقريباً")
if du.free / 1e9 < need_gb:
    print(f"        [تحذير] المساحة الحرة ({du.free/1e9:.1f}GB) قد لا تكفي!")
    print("                خفّض YODAS_HOURS، أو وسّع القرص الشبكي، أو فعّل CLEAN_RAW_AFTER (الخلية 24).")
else:
    print("        [جيد] المساحة الحرة كافية.")

# ---- الأدوات ----
for tool in ("ffmpeg", "ffprobe", "git", "curl", "uv"):
    print(f"[أداة] {tool:<8}: {'متوفر' if shutil.which(tool) else 'غير متوفر'}")

print("\n[فحص مبدئي] أكمل إلى الخلية 3 (التبعيات).")


In [ ]:
# ============================================================ #
#  الخلية ٣ — تبعيات النظام (ffmpeg / git) + مدير الحزم uv     #
#  (حاويات RunPod تعمل بصلاحيات root عادة — بلا sudo)          #
# ============================================================ #
import shutil, os, subprocess
from pathlib import Path

def have(x):
    return shutil.which(x) is not None

missing = [p for p in ("ffmpeg", "git") if not have(p)]
if missing:
    print(f"تثبيت حزم النظام: {', '.join(missing)} ...")
    sh(["apt-get", "update", "-qq"], check=False)
    sh(["apt-get", "install", "-y", "-qq", *missing])
for tool in ("ffmpeg", "git", "curl"):
    print(f"  {tool:<8}: {'متوفر' if have(tool) else 'غير متوفر!'}")

# ---- uv: مُسرّع تثبيت بايثون (نفس مسار سكربتات المستودع) ----
UV = shutil.which("uv")
if UV is None:
    print("\nتثبيت uv (مرة واحدة) ...")
    sh(["bash", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=False)
    for cand in (Path.home() / ".local" / "bin" / "uv", Path.home() / ".cargo" / "bin" / "uv"):
        if cand.exists():
            UV = str(cand)
            break
    if UV:
        os.environ["PATH"] = str(Path(UV).parent) + os.pathsep + os.environ["PATH"]
        ENV["PATH"] = os.environ["PATH"]
if UV:
    r = subprocess.run([UV, "--version"], capture_output=True, text=True)
    print(f"\nuv جاهز: {r.stdout.strip() or r.stderr.strip()}")
else:
    print("\nuv غير متوفر — سنعتمد python3 -m venv + pip كبديل في الخلية 4 (يعمل أيضاً).")


In [ ]:
# ============================================================ #
#  الخلية ٤ — بيئة Python المعزولة + PyTorch 2.4.0 (CUDA 12.1) #
#  نفس الإصدارات المثبتة في سكربتات المستودع الرسمية            #
#  (تُبنى مرة واحدة على القرص الدائم وتُعاد استخدامها دائماً)    #
# ============================================================ #
import subprocess, shutil, sys
from pathlib import Path

UV = globals().get("UV") or shutil.which("uv")   # تحصين: يعمل حتى بعد إعادة تشغيل النواة

def venv_ok(code):
    # فحص سريع داخل البيئة المعزولة
    if not VENV_PY.exists():
        return False
    r = subprocess.run([str(VENV_PY), "-c", code], capture_output=True, text=True, env=ENV)
    return r.returncode == 0

# ---- 1) إنشاء venv إن لم يوجد ----
if not VENV_PY.exists():
    print(f"إنشاء بيئة معزولة (Python 3.11) في {ROOT/'venv'} ...")
    if UV:
        sh([UV, "venv", "--python", "3.11", str(ROOT / "venv")])
    else:
        sh([sys.executable, "-m", "venv", str(ROOT / "venv")])

# ---- 2) PyTorch 2.4.0 cu121 (نحو 2.5GB — يُتخطى إن كان مثبتاً) ----
if not venv_ok("import torch; assert torch.__version__.startswith('2.4.')"):
    print("تثبيت PyTorch 2.4.0 + torchaudio (cu121) — عدة دقائق حسب الشبكة ...")
    trio_pin = ["torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0"]
    trio_any = ["torch", "torchvision", "torchaudio"]
    idx = ["--index-url", "https://download.pytorch.org/whl/cu121"]
    if UV:
        rc = sh([UV, "pip", "install", "--python", str(VENV_PY), *trio_pin, *idx], check=False)
        if rc != 0:
            sh([UV, "pip", "install", "--python", str(VENV_PY), *trio_any, *idx])
    else:
        rc = sh([str(VENV_PY), "-m", "pip", "install", "-q", *trio_pin, *idx], check=False)
        if rc != 0:
            sh([str(VENV_PY), "-m", "pip", "install", "-q", *trio_any, *idx])
else:
    print("PyTorch 2.4.x موجود أصلاً في البيئة — تخطّي التثبيت.")

# ---- 3) بقية الاعتماديات (مثبتة بالإصدارات نفسها التي نجح بها خط المستودع) ----
NEED = ("import torch, torchaudio, librosa, soundfile, munch, einops,"
        " transformers, pyarrow, huggingface_hub, yaml, matplotlib")
if not venv_ok(NEED):
    print("تثبيت الاعتماديات (numpy/scipy/librosa/transformers/...) ...")
    pins = ["numpy==1.26.4", "scipy==1.13.1", "librosa==0.10.2", "soundfile==0.12.1",
            "munch==4.0.0", "einops==0.8.0", "transformers==4.46.3", "pyyaml", "tqdm",
            "requests", "pyarrow==17.0.0", "huggingface_hub", "matplotlib"]
    loose = ["numpy", "scipy", "librosa", "soundfile", "munch", "einops", "transformers",
             "pyyaml", "tqdm", "requests", "pyarrow", "huggingface_hub", "matplotlib"]
    if UV:
        rc = sh([UV, "pip", "install", "--python", str(VENV_PY), *pins], check=False)
        if rc != 0:
            sh([UV, "pip", "install", "--python", str(VENV_PY), *loose])
    else:
        rc = sh([str(VENV_PY), "-m", "pip", "install", "-q", *pins], check=False)
        if rc != 0:
            sh([str(VENV_PY), "-m", "pip", "install", "-q", *loose])
else:
    print("الاعتماديات موجودة أصلاً — تخطّي التثبيت.")

# ---- 4) تحقق نهائي (بما فيه رؤية GPU من داخل البيئة) ----
r = subprocess.run([str(VENV_PY), "-c",
                    "import torch, torchaudio; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"],
                   capture_output=True, text=True, env=ENV)
print((r.stdout + r.stderr).strip())
assert r.returncode == 0 and "torch 2.4" in (r.stdout + r.stderr), "فشل بناء البيئة — راجع السجل أعلاه"
print(f"\n[جاهز] البيئة المعزولة: {VENV_PY}")


In [ ]:
# ============================================================ #
#  الخلية ٥ — استنساخ المستودعات + بصمات قابلية إعادة الإنتاج   #
#  Seed-VC (المدرب الرسمي) + رافِدَين (خط البيانات العراقية)    #
#  (على القرص الدائم — يُحدثان إن وُجدا بدلاً من إعادة التنزيل) #
# ============================================================ #
import subprocess, json
from pathlib import Path

def git_clone(url, dst):
    if (dst / ".git").exists():
        print(f"تحديث {dst.name} ...")
        sh(["git", "-C", str(dst), "pull", "--ff-only"], check=False)
    else:
        print(f"استنساخ {dst.name} من {url} ...")
        sh(["git", "clone", "--depth", "1", url, str(dst)])

git_clone("https://github.com/Plachtaa/seed-vc.git", SEEDVC)
git_clone("https://github.com/Alqudimi/rafidain.git", RAFIDAIN)

# تحقق من الملفات الحرجة
for f in ("train.py", "inference.py", "configs/presets/config_dit_mel_seed_uvit_xlsr_tiny.yml"):
    assert (SEEDVC / f).exists(), f"ملف مفقود في seed-vc: {f}"
assert (RAFIDAIN / "seed_vc_iraqi_ubuntu.sh").exists(), "سكربت خط البيانات مفقود في rafidain"

# ---- بصمات الإصدارات (تُحفظ مع حزمة النموذج لاحقاً — قابلية إعادة الإنتاج) ----
def git_hash(path):
    try:
        r = subprocess.run(["git", "-C", str(path), "rev-parse", "--short", "HEAD"],
                           capture_output=True, text=True, timeout=30)
        return r.stdout.strip() or "?"
    except Exception:
        return "?"

repro = {
    "notebook_version": NB_VERSION,
    "seedvc_commit": git_hash(SEEDVC),
    "rafidain_commit": git_hash(RAFIDAIN),
    "captured_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(ROOT / "repro.json").write_text(json.dumps(repro, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[بصمات] seed-vc @ {repro['seedvc_commit']} | rafidain @ {repro['rafidain_commit']}")
print(f"[جاهز] seed-vc   : {SEEDVC}")
print(f"[جاهز] rafidain  : {RAFIDAIN}")


In [ ]:
# ============================================================ #
#  الخلية ٦ — استخراج خط بيانات العراق المدمج                   #
#  (الكتالوج العراقي كله مضمّن داخل سكربت المستودع نفسه:         #
#   49,194 مقطعاً / 339 ساعة / خريطة إزاحات بايت-دقيقة           #
#   داخل أرشيفات espnet/yodas3 — نستخرجه هنا كوحدة مستقلة)      #
# ============================================================ #
import re, gzip, base64, json, ast, urllib.request
from pathlib import Path

SRC_SH = RAFIDAIN / "seed_vc_iraqi_ubuntu.sh"
if not SRC_SH.exists():   # احتياط: المستودع غير منسوخ — ننزّل السكربت مباشرة من GitHub
    print("تنزيل السكربت من GitHub مباشرة ...")
    SRC_SH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Alqudimi/rafidain/main/seed_vc_iraqi_ubuntu.sh", SRC_SH)

text = SRC_SH.read_text(encoding="utf-8")
m = re.search(r"cat > \"\$ROOT/iraqi_pipeline\.py\" <<'PYEOF'\n(.*?)\nPYEOF\n", text, re.S)
assert m, "لم يُعثر على خط البيانات المدمج داخل السكربت!"
code = m.group(1)
ast.parse(code)   # تحقق من سلامة الكود المستخرج قبل الكتابة
PIPE.write_text(code, encoding="utf-8")
print(f"[خط البيانات] استُخرج إلى: {PIPE}  ({len(code):,} بايت)")

# ---- إحصاءات الكتالوج العراقي المدمج (بدون أي تنزيل) ----
def _grab(name):
    blob = re.search(name + r' = r"""(.*?)"""', code, re.S).group(1)
    return json.loads(gzip.decompress(base64.b64decode(blob)))

clips, offs, srcrank = _grab("_IQ_CLIPS_B64"), _grab("_IQ_OFFS_B64"), _grab("_IQ_SRC_B64")
hours_total = sum(c["du"] for c in clips) / 3600
resolvable = [s for s, v in offs.items() if v.get("off")]
print(f"[كتالوج اللهجة العراقية] {len(clips):,} مقطعاً | {hours_total:.0f} ساعة | "
      f"{len(resolvable):,} تسجيل مصدر قابل للجلب | {sum(offs[s]['size'] for s in resolvable)/1e9:.1f}GB لو جُلب كله")

# ---- حجم هدفك الحالي (نفس منطق الاختيار في السكربت: الجودة أولاً) ----
t = str(CFG["YODAS_HOURS"]).strip().lower()
if t in ("0", ""):
    print(f"\n[هدفك] YODAS معطّل (YODAS_HOURS=0) — الخلية 8 ستتخطاه. الأصل المتاح: {hours_total:.0f} ساعة.")
else:
    target = None if t in ("all", "كل") else float(t)
    ranked = sorted(srcrank.items(), key=lambda kv: -kv[1][2])
    chosen, h = [], 0.0
    for sid, (tar, ch, q) in ranked:
        if sid not in offs or offs[sid].get("off") is None:
            continue
        chosen.append(sid); h += ch
        if target and h >= target:
            break
    size_gb = sum(offs[s]["size"] for s in chosen) / 1e9
    label = f"{target:.0f} ساعة" if target else "الكتالوج كله"
    print(f"[هدفك] {label}: {len(chosen):,} تسجيلاً مصدراً ≈ {size_gb:.1f}GB تنزيلاً خاماً (سيُشفَّر إلى Opus للتدريب)")
print("\n[جاهز] تابع إلى خلايا البيانات (7 وما بعدها).")


<div dir="rtl">

## مرحلة البيانات — المصادر

| المصدر | ما هو | الحجم | ملاحظات |
|---|---|---|---|
| `yodas` | كلام عراقي من كتالوج YODAS-v3 الموسوم (dialect=iq): 49,194 مقطعاً / 339 ساعة عبر 2,360 تسجيلاً | حسب ساعاتك | الأكبر والأهم — جلب ذكي بإزاحات بايت من أرشيفات espnet بلا تنزيل الأرشيف كاملاً، مرتّب بترتيب الجودة |
| `youssef` | 3,867 مقطعاً عراقياً 22.05kHz مع نصوص | ~720MB | نظيف — مناسب أيضاً كمرجع استنساخ |
| `shiekh` | 827 مقطعاً لمتحدث ديني واحد 16kHz | ~640MB | يعمّق التنوع الأسلوبي |
| `safaa` | 676 عينة لصوت أنثوي عراقي (جاهزة TTS) | صغير | توازن أنثوي مهم للاستنساخ النسائي |
| `lahgtna` | الجزء العراقي من Lahgtna v2: ~10,746 مقطعاً / ≈50 ساعة 16kHz مع نصوص لهجية | ≈5.8GB | جديد في ٢.١ — يجلب عبر datasets-server بترقيم صفحات كامل؛ قد يتداخل جزئياً مع مصادر YODAS |
| `kharrufa` | 450 مقطعاً استوديو للهجة العراقية 44.1kHz بنصوص مشكولة كاملة | ≈2.1GB تنزيلاً | جديد في ٢.١ — Iraqi Dialect TTS Corpus (CC BY 4.0)؛ يُستخرج 683MB ثم يُحذف الأرشيف |
| `custom` | **تسجيلاتك أنت** (اختيارية) | حسبك | الخلية 9 — أعلى أثر لصوتك المستهدف |

يضاف فوقها في الإصدار ٢.١ من الدفتر: ≈50 ساعة عراقية إضافية من Lahgtna v2 (الصفوف الموسومة iq) وساعة استوديو نظيفة من corpus الخرّوفا — الخلية 7 تجلبها تلقائياً مع المجموعات الصغيرة.

**سياسة الجودة:** الكتالوج يرتّب المصادر بدرجة جودة محسوبة مسبقاً، والجلب يبدأ من الأفضل — أي أن «200 ساعة» تعني أفضل 200 ساعة متاحة، لا 200 عشوائية.

> **أخلاقيات:** هذه بيانات كلام عامة من أرشيفات بحثية مفتوحة. عند استنساخك صوت شخص حقيقي في الإنتاج، موافقته الصريحة الموثقة إلزامية.

</div>


In [ ]:
# ============================================================ #
#  الخلية ٧ — تنزيل المجموعات العراقية النظيفة                  #
#  youssef / shiekh / safaa (HuggingFace)                       #
#  + lahgtna: ≈50 ساعة عراقية من Lahgtna v2 (الجزء الأكبر)     #
#    عبر datasets-server بترقيم صفحات كامل                     #
#  + kharrufa: 450 مقطع استوديو من Google Drive                #
#  قابلة للاستئناف: أعد التشغيل لتكملة ما فشل فقط              #
# ============================================================ #
if CFG["SKIP_SMALL"]:
    print("تخطي المجموعات النظيفة الخمس (SKIP_SMALL=True) — إن أردت فعّلها من الخلية 1.")
else:
    import shutil
    free0 = shutil.disk_usage(ROOT).free / 1e9
    print(f"المساحة الحرة قبل التنزيل: {free0:.1f}GB (المرحلة كلها ≈7.5GB، أكبر عنصر فيها lahgtna)")
    sh([VENV_PY, PIPE, "small",
        "--root", ROOT,
        "--workers", int(CFG["FETCH_WORKERS"])])
    AUD = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".webm"}
    for name, note in (("youssef", "مقاطع قصيرة نظيفة"),
                       ("shiekh", "متحدث ديني واحد"),
                       ("safaa", "صوت أنثى عراقي"),
                       ("lahgtna", "≈50 ساعة عراقية إضافية — جديد ٢.١"),
                       ("kharrufa", "450 مقطع استوديو — جديد ٢.١")):
        d = ROOT / "raw" / name
        n = len([p for p in d.iterdir() if p.suffix.lower() in AUD]) if d.exists() else 0
        print(f"  {name:<8}: {n:,} ملفاً | {note}")
    free1 = shutil.disk_usage(ROOT).free / 1e9
    print(f"استُهلك {free0-free1:.1f}GB | الحر الآن {free1:.1f}GB")
    st = ROOT / "raw" / "_lahgtna_state.json"
    if st.exists():
        import json
        s = json.loads(st.read_text())
        fp, fa = len(s.get("failed_pages", [])), len(s.get("failed_audio", []))
        if fp or fa:
            print(f"[ملاحظة] lahgtna: {fp} صفحة و{fa} مقطعاً فشلا مؤقتاً — "
                  "أعد هذه الخلية لإعادة المحاولة فقط (الموجود لا يُعاد)")
print("\n[جاهز] تابع إلى الخلية 8 (YODAS) — أو الخلية 9 لدمج ملفاتك الخاصة.")


In [ ]:
# ============================================================ #
#  الخلية ٨ — جلب بيانات YODAS العراقية (الكتلة الكبرى)        #
#  تنزيل مُجزَّأ متوازٍ من أرشيفات espnet/yodas3                #
#  يستغرق عادة 15-60 دقيقة (حسب الشبكة وساعاتك)                #
#  قابلاً للاستئناف 100%: أعد التشغيل ليكمل من حيث توقف         #
# ============================================================ #
import shutil

t = str(CFG["YODAS_HOURS"]).strip().lower()
if t == "0":
    print("YODAS معطّل (YODAS_HOURS=0) — تخطّي.")
else:
    free0 = shutil.disk_usage(ROOT).free / 1e9
    print(f"المساحة الحرة قبل الجلب: {free0:.1f}GB")
    sh([VENV_PY, PIPE, "yodas",
        "--root", ROOT,
        "--hours", str(CFG["YODAS_HOURS"]),
        "--workers", int(CFG["FETCH_WORKERS"])])
    n = len(list((ROOT / "raw" / "yodas_webm").glob("*.webm"))) if (ROOT / "raw" / "yodas_webm").exists() else 0
    free1 = shutil.disk_usage(ROOT).free / 1e9
    print(f"\nتسجيلات مصدر جاهزة: {n:,} ملف webm | استُهلك {free0-free1:.1f}GB | الحر الآن {free1:.1f}GB")
    state = ROOT / "raw" / "yodas_fetch_state.json"
    if state.exists():
        import json
        failed = json.loads(state.read_text()).get("failed", [])
        if failed:
            print(f"مصادر فشل جلبها: {len(failed)} — أعد هذه الخلية لإعادة محاولة المتبقي، أو تجاهل (بقية البيانات تكفي)")
print("\n[جاهز] تابع إلى الخلية 9 (اختيارية) ثم 10 (التقطيع والترميز).")


In [ ]:
# ============================================================ #
#  الخلية ٩ — (اختيارية) دمج تسجيلاتك الخاصة في التدريب         #
#  ارفع ملفاتك (wav/flac/mp3/m4a/opus/ogg) إلى:                #
#     /workspace/iraqi_seedvc/raw/custom/                      #
#  عبر متصفح ملفات Jupyter (سحب وإفلات) ثم أعد تشغيل الخلية    #
#  ستُقطَّع تلقائياً إلى مقاطع 1-29.5 ثانية وتُرمَّز Opus 24kHz   #
# ============================================================ #
import sys, os, json, time
from multiprocessing import Pool
from pathlib import Path

CUSTOM = ROOT / "raw" / "custom"
CUSTOM.mkdir(parents=True, exist_ok=True)
EXTS = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".webm"}
files = [p for p in sorted(CUSTOM.iterdir()) if p.is_file() and p.suffix.lower() in EXTS]
print(f"مجلد بياناتك: {CUSTOM}")
print(f"ملفات صوتية موجودة: {len(files)}")

if not files:
    print('''
لا توجد ملفات بعد — لتفعيل ذلك:
  1) ارفع تسجيلاتك إلى المجلد أعلاه (سحب وإفلات في لوحة ملفات Jupyter)
  2) الأفضل: كلام نظيف بلا موسيقى أو ضجيج، 1-30 ثانية أو أطول (تُقطَّع تلقائياً)
  3) أعد تشغيل هذه الخلية، ثم شغّل خلية التقطيع (10) لتشمل بياناتك
(تخطَّ هذه الخلية إن لم يكن لديك بيانات خاصة — مجموعة YODAS + الصغيرة كافية تماماً)''')
else:
    sys.path.insert(0, str(ROOT))
    import iraqi_pipeline as P   # خط البيانات المستخرج في الخلية 6 — نفس منطق التقطيع المجرَّب
    import types
    a = types.SimpleNamespace(out=str(DATA_DIR))
    jobs = []
    for f in files:
        jobs.extend(P._prep_job_src(a, str(f), "custom"))
    print(f"سيُنتج عنها {len(jobs)} مقطعاً (يُتخطى الموجود منها)")
    t0, ok_n, bad_n = time.time(), 0, 0
    with Pool(processes=int(CFG["PREP_WORKERS"] or 0) or min(16, os.cpu_count() or 4)) as pool:
        for out, dur, okk in pool.imap_unordered(P._run_prep_job, jobs, chunksize=16):
            if okk:
                ok_n += 1
            else:
                bad_n += 1
    print(f"تم: {ok_n} مقطعاً ناجحاً | {bad_n} فاشلاً في {time.time()-t0:.0f} ثانية")
    # تحديث فهرس بيانات التدريب (كما يفعل أمر prep نفسه)
    idx = DATA_DIR / "_prep_index.jsonl"
    prev = {}
    if idx.exists():
        for line in idx.read_text().splitlines():
            try:
                r = json.loads(line)
                prev[r["file"]] = r["dur"]
            except Exception:
                pass
    for j in jobs:
        prev[os.path.relpath(j[3], DATA_DIR)] = j[2]
    prev = {f: d for f, d in prev.items() if (DATA_DIR / f).exists()}
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    idx.write_text("".join(json.dumps({"file": f, "dur": d}) + "\n" for f, d in sorted(prev.items())))
    print("فهرس بيانات التدريب محدّث — أكمل إلى الخلية 10.")


In [ ]:
# ============================================================ #
#  الخلية ١٠ — التقطيع والترميز: بناء بيانات التدريب النهائية   #
#  كل المقاطع -> Opus 24kHz أحادي، 1-29.5 ثانية                #
#  (يشمل YODAS + المجموعات الخمس + بياناتك إن وضعتها)          #
#  قابلة للاستئناف: الموجود يُتخطى                              #
# ============================================================ #
import os

workers = int(CFG["PREP_WORKERS"] or 0) or min(16, os.cpu_count() or 4)
sh([VENV_PY, PIPE, "prep",
    "--root", ROOT,
    "--out", DATA_DIR,
    "--workers", workers])

n = sum(1 for _ in DATA_DIR.rglob("*.ogg"))
print(f"\nمقاطع التدريب الجاهزة: {n:,} ملف Opus في {DATA_DIR}")
assert n >= 50, f"مقاطع قليلة جداً ({n}) — تأكد أن الخليتين 7 و8 أنزلتا بيانات ثم أعد المحاولة"
print("[جاهز] البيانات مبنية — تابع إلى الخلية 11 (تقرير البيانات) ثم 12 (عزل الاختبار).")


In [ ]:
# ============================================================ #
#  الخلية ١١ — تقرير بيانات التدريب (جديد في الإصدار ٢)        #
#  ساعات كل مصدر + مدرّج أطوال المقاطع + فحص جودة سريع        #
#  (تسميات الرسم بالإنجليزية لأن matplotlib لا يشكّل            #
#   الحروف العربية المتصلة بشكل صحيح)                          #
# ============================================================ #
import json
from pathlib import Path

# ---- إحصاء من فهرس التجهيز + من الملفات الفعلية ----
idx = DATA_DIR / "_prep_index.jsonl"
rows = []
if idx.exists():
    for line in idx.read_text().splitlines():
        try:
            r = json.loads(line)
            if (DATA_DIR / r["file"]).exists():
                rows.append(r)
        except Exception:
            pass

per, tot_h, tot_n = {}, 0.0, 0
for r in rows:
    src = r["file"].split("/")[0] if "/" in r["file"] else "(جذر)"
    per.setdefault(src, [0, 0.0])
    per[src][0] += 1
    per[src][1] += float(r["dur"]) / 3600
    tot_n += 1
    tot_h += float(r["dur"]) / 3600

print("=" * 52)
print("تقرير بيانات التدريب المجهزة")
print("=" * 52)
for k in sorted(per):
    print(f"  {k:<10}: {per[k][0]:6,} مقاطع | {per[k][1]:7.2f} ساعة")
print(f"  {'الإجمالي':<10}: {tot_n:6,} مقاطع | {tot_h:7.2f} ساعة")
print("=" * 52)

if tot_h < 1:
    print("[تحذير] أقل من ساعة واحدة — التدريب سيكون ضعيفاً. زد YODAS_HOURS أو أضف بياناتك.")
elif tot_h < 10:
    print("[ملاحظة] أقل من 10 ساعات — يكفي لتجربة المسار، ولجودة إنتاجية زد البيانات.")
else:
    print(f"[جيد] {tot_h:.0f} ساعة عراقية — جرعة تدريب قوية.")

# ---- مدرّج أطوال المقاطع ----
if rows:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    durs = [float(r["dur"]) for r in rows]
    fig, ax = plt.subplots(figsize=(9, 3.4), constrained_layout=True)
    ax.hist(durs, bins=60, color="#1f77b4", edgecolor="white", linewidth=0.4)
    ax.axvline(sum(durs) / len(durs), color="#d62728", lw=1.6,
               label=f"mean = {sum(durs)/len(durs):.1f}s")
    ax.set_xlabel("segment duration (s)")
    ax.set_ylabel("segments")
    ax.set_title(f"Iraqi training segments - {tot_n:,} files / {tot_h:.1f} h")
    ax.legend()
    ax.grid(alpha=0.3)
    chart = ROOT / "data" / "duration_hist.png"
    fig.savefig(chart, dpi=110)
    plt.close(fig)
    from IPython.display import display, Image as IPyImage
    display(IPyImage(filename=str(chart)))
    print(f"(حُفظ بشكل دائم: {chart} — سيُرفق بحزمة النموذج)")
    # فحص جودة: نسبة المقاطع خارج نطاق 1.5-29.5 ث
    bad = sum(1 for d in durs if d < 1.5 or d > 29.9)
    print(f"مقاطع خارج النطاق المثالي: {bad} ({100*bad/len(durs):.1f}%) — ستُتخطى تلقائياً في التدريب")


In [ ]:
# ============================================================ #
#  الخلية ١٢ — عزل بيانات الاختبار (جديد في الإصدار ٢)         #
#  نستبعد عدداً من المقاطع العراقية من التدريب نهائياً،         #
#  لتقييم النموذج لاحقاً على كلام لم تره عيناه إطلاقاً          #
#  (التقييم العلمي الوحيد المعتدل — وإلا فأنت تختبر الحفظ)     #
# ============================================================ #
import json, random
from pathlib import Path

TEST_DIR.mkdir(parents=True, exist_ok=True)
hl_path = ROOT / "data" / "holdout_list.json"

# 1) اختيار حتمي (نفس البذرة دائماً = نفس القائمة بين الجلسات)
idx = DATA_DIR / "_prep_index.jsonl"
rows = []
if idx.exists():
    for line in idx.read_text().splitlines():
        try:
            r = json.loads(line)
            if (DATA_DIR / r["file"]).exists():
                rows.append(r)
        except Exception:
            pass
assert rows, "لا يوجد فهرس بيانات — شغّل الخلية 10 (التقطيع) أولاً"

if hl_path.exists():
    holdout = json.loads(hl_path.read_text())
    print(f"قائمة اختبار موجودة: {len(holdout)} مقطعاً (من اختيار سابق — تُحترم كما هي)")
else:
    rng = random.Random(42)   # بذرة ثابتة: قابلية إعادة إنتاج كاملة
    # نفضل مقاطع 3-15 ثانية (مثالية كعينات تقييم: ليست قصيرة جداً ولا ضخمة)
    pool = [r for r in rows if 3.0 <= float(r["dur"]) <= 15.0]
    if len(pool) < CFG["HOLDOUT_COUNT"]:
        pool = rows
    take = min(int(CFG["HOLDOUT_COUNT"]), len(pool))
    # تنويع المصادر: نوزع الاختيار بالتساوي على المصادر المتاحة
    by_src = {}
    for r in pool:
        by_src.setdefault(r["file"].split("/")[0] if "/" in r["file"] else "root", []).append(r)
    for lst in by_src.values():
        rng.shuffle(lst)
    holdout, i = [], 0
    while len(holdout) < take:
        added = False
        for lst in by_src.values():
            if i < len(lst) and len(holdout) < take:
                holdout.append(lst[i]["file"])
                added = True
        if not added:
            break
        i += 1
    hl_path.write_text(json.dumps(holdout, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"قائمة اختبار جديدة: {len(holdout)} مقطعاً (بذرة ثابتة 42 — موزعة على المصادر)")

# 2) تنفيذ العزل الفعلي (idempotent: يعمل بأمان أي عدد من المرات)
moved = kept = 0
for rel in holdout:
    src = DATA_DIR / rel
    dst = TEST_DIR / Path(rel).name
    if src.exists():
        if dst.exists():
            src.unlink()          # موجود في جهة الاختبار أصلاً — نحذف نسخة التدريب
        else:
            src.rename(dst)
        moved += 1
    elif dst.exists():
        kept += 1

print(f"عُزل الآن: {moved} | كان معزولاً سابقاً: {kept} | إجمالي الاختبار: {len(list(TEST_DIR.glob('*.ogg')))} مقطعاً")
print(f"مجلد الاختبار: {TEST_DIR}")

# 3) تحديث إحصاءات التدريب بعد الاستبعاد (يحذف من الفهرس المداخل غير الموجودة)
sh([VENV_PY, PIPE, "stats", "--dir", DATA_DIR], check=False)

# ملاحظة: إذا أعدت تشغيل الخلية 10 (prep) لاحقاً فقد تعود بعض المقاطع لمجلد التدريب —
# لا مشكلة: خلية إطلاق التدريب (14) تعيد العزل تلقائياً قبل الانطلاق (reapply_holdout).
print("\n[جاهز] تابع إلى الخلية 13 (تنزيل نقاط النموذج).")


In [ ]:
# ============================================================ #
#  الخلية ١٣ — تنزيل نقاط النموذج الأساسية مسبقاً (≈2.2GB)      #
#  DiT_uvit_tat_xlsr_ema + XLSR-300m + HiFT + CAMPPlus +       #
#  OpenVoice — كلها في مجلد checkpoints داخل seed-vc           #
#  (يمنع تعطل بدء التدريب لاحقاً بسبب التنزيل، وقابل للاستئناف) #
# ============================================================ #
sh([VENV_PY, PIPE, "prefetch", "--repo", SEEDVC])
import shutil
ck = SEEDVC / "checkpoints"
total = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e9 if ck.exists() else 0
print(f"\nحجم مخزون النقاط: {total:.2f}GB في {ck}")

# حارس: تأكيد النقطة الأم التي يبدأ منها التدريب (DiT_uvit_tat_xlsr_ema.pth)
base_ema = list(ck.rglob("DiT_uvit_tat_xlsr_ema.pth")) if ck.exists() else []
assert base_ema, "نقطة الأساس DiT_uvit_tat_xlsr_ema.pth غير موجودة بعد التنزيل — أعد هذه الخلية"
print(f"[مؤكد] نقطة الأساس جاهزة: {base_ema[0]}")
print("[جاهز] كل شيء مطلوب للتدريب موجود — تابع إلى مرحلة التدريب.")


<div dir="rtl">

## مرحلة التدريب — كيف تعمل ولماذا هي صحيحة؟

- **نقطة البداية دائماً** الأوزان المنشورة `DiT_uvit_tat_xlsr_ema.pth` من `Plachta/Seed-VC` — أي أننا نضبط دقيقاً نموذجاً مدرباً مسبقاً بجدارة، **لا ندرب من الصفر**. لهذا تحافظ المخرجات على خاصية الاستنساخ اللحظي zero-shot وتتخصص للعراقي في آن واحد.
- **الكود المدرب هو `train.py` الرسمي** من مستودع Seed-VC نفسه — بلا أي تعديل — بنفس معدل التعلم المعتمد upstream (1e-5، warmup=0) الذي أثبت نفسه في توليف النموذج.
- **آلية الاستئناف المدمجة:** إن وُجدت نقاط `DiT_epoch_*_step_*.pth` سابقة في مجلد التشغيل، يبدأ منها تلقائياً؛ وإلا فيبدأ من نقطة الأساس المنشورة. (ملاحظة تقنية: عند الاستئناف يُعاد عدّاد الخطوات من صفر لكن الأوزان تكمل من آخر نقطة — لذلك مجموع الخطوات الفعلي قد يتجاوز `STEPS` قليلاً عبر جلسات متعددة، وهذا آمن ومقصود).
- **عملية مستقلة تماماً** (`start_new_session`): إغلاق المتصفح أو انقطاع الشبكة أو إعادة تشغيل نواة الدفتر لا يمسها. سجلها الحي في `runs/<اسم-التشغيل>/train_console.log`.
- **النقاط الوسطية** تُحفظ كل `SAVE_EVERY` خطوة ويُبقى على آخر نقطتين (سلوك المدرب الرسمي) — تستطيع مقارنتها لاحقاً.
- **الناتج النهائي** `ft_model.pth` يُكتب تلقائياً عند اكتمال الخطوات.

### الجرعة الموصى بها (مطابقة لدليل المستودع FINETUNING.md)

| هدفك | STEPS | الزمن التقريبي (RTX 4090) |
|---|---|---|
| تكيّف عام مع اللهجة (الموصى به) | 3000 | ~25–40 دقيقة |
| جرعة أولى سريعة للتحقق من المسار | 1000 | ~10 دقائق |
| تعميق أقصى (بيانات كبيرة) | 4000–6000 | ~1–1.5 ساعة |

> القاعدة: راقب منحنى الخسارة في خلية المراقبة — إذا استوى تماماً فزاد الخطوات لن يفيد، وإذا كان ينزل فالمجال مفتوح. وكل نقطة حفظ يمكنك البناء فوقها لاحقاً بتغيير `RUN_NAME` وتشغيل جديد يبدأ من `--pretrained-ckpt` بالنقطة السابقة.

</div>


In [ ]:
# ============================================================ #
#  الخلية ١٤ — التدريب المباشر (استيراد train.py كوحدة بايثون)   #
#  استدعاء مباشر لـ Trainer.train() بلا subprocess            #
#  التقدم يظهر مباشرة في الخلية ويُحفظ في train_console.log     #
# ============================================================ #
import os, sys, time, json, io, contextlib
from pathlib import Path
import torch

LOG = RUN_DIR / "train_console.log"
FT = RUN_DIR / "ft_model.pth"
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

# ---- حراس قبل الانطلاق ----
assert torch.cuda.is_available(), "لا يوجد GPU متاح للنواة! التدريب يتطلب حاوية GPU على RunPod."
assert (SEEDVC / "train.py").exists(), "مستودع seed-vc غير موجود — شغّل الخلية 5"
assert DATA_DIR.exists() and sum(1 for _ in DATA_DIR.rglob("*.ogg")) >= 50, \
    "بيانات تدريب غير كافية — شغّل خلايا البيانات (7-10)"
_n_test = len(list(TEST_DIR.glob("*.ogg"))) if TEST_DIR.exists() else 0
if _n_test == 0:
    print("[تنبيه] لا توجد بيانات اختبار معزولة — خلية التقييم A/B (18) ستعمل على عينة من بيانات التدريب.")

# ---- منع تسريب بيانات الاختبار ----
reapply_holdout()

BATCH = int(CFG.get("_BATCH") or 2)
MAXS = int(CFG["STEPS"])
SAVE_EVERY = int(CFG["SAVE_EVERY"])
WORKERS = int(CFG.get("TRAIN_WORKERS", 2))

if FT.exists():
    print(f"النموذج النهائي موجود أصلاً: {FT}")
    print("لتدريب جديد كلياً: غيّر RUN_NAME في الخلية 1 (أو احذف ft_model.pth ثم أعد هذه الخلية).")
else:
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # التأكد من إدراج مسارات seed-vc في sys.path للاستيراد المباشر
    for sp in (SEEDVC, SEEDVC / "modules"):
        if str(sp) not in sys.path:
            sys.path.insert(0, str(sp))

    import train as seed_train

    class Tee(io.TextIOBase):
        def __init__(self, original, path):
            self.original = original
            self.file = open(path, "a", encoding="utf-8", buffering=1)
        def write(self, s):
            self.original.write(s)
            self.file.write(s)
            return len(s)
        def flush(self):
            self.original.flush()
            self.file.flush()

    t0 = time.time()
    print(f"[train] بدء التدريب باستدعاء seed_train.Trainer مباشرة: {MAXS} خطوة (دفعة {BATCH}، حفظ كل {SAVE_EVERY})")
    print(f"  السجل الكامل: {LOG}")
    print(f"  مجلد النقاط  : {RUN_DIR}")

    _old_cwd = os.getcwd()
    os.chdir(str(SEEDVC))
    try:
        with contextlib.redirect_stdout(Tee(sys.stdout, str(LOG))):
            trainer = seed_train.Trainer(
                config_path=str(CFG_TINY),
                pretrained_ckpt_path=None,
                data_dir=str(DATA_DIR),
                run_name=CFG["RUN_NAME"],
                batch_size=BATCH,
                num_workers=WORKERS,
                steps=MAXS,
                save_interval=SAVE_EVERY,
                max_epochs=1000,
                device="cuda:0",
            )
            trainer.train()
    finally:
        os.chdir(_old_cwd)

    mins = (time.time() - t0) / 60
    if FT.exists():
        print(f"\n[اكتمل] انتهى التدريب بنجاح في {mins:.1f} دقيقة — النموذج النهائي: {FT}")
    else:
        print(f"\n[تحذير] توقف التدريب بعد {mins:.1f} دقيقة — راجع السجل: {LOG}")


In [ ]:
# ============================================================ #
#  الخلية ١٥ — مراقبة التدريب (أعد تشغيلها متى شئت)            #
#  تقدم + معدل + زمن متبقٍ + منحنى الخسارة + آخر السجل          #
#  + حارس NaN: يكشف انهيار الخسارة فوراً (جديد في الإصدار ٢)    #
# ============================================================ #
import re, os, time
from pathlib import Path
from IPython.display import display, Image as IPyImage

LOG = RUN_DIR / "train_console.log"
PIDF = RUN_DIR / "train.pid"
FT = RUN_DIR / "ft_model.pth"
STEP_RE = re.compile(r"epoch (\d+), step (\d+), loss: ([0-9.eEnaif+-]+)")
MAXS = int(CFG["STEPS"])

def _alive():
    try:
        if PIDF.exists():
            os.kill(int(PIDF.read_text().strip()), 0)
            return True
    except Exception:
        pass
    return False

lines = LOG.read_text(errors="replace").splitlines() if LOG.exists() else []
pts = []
for ln in lines:
    m = STEP_RE.search(ln)
    if m:
        try:
            pts.append((int(m.group(2)), float(m.group(3))))
        except ValueError:
            pass

if FT.exists():
    status = "منجز — ft_model.pth موجود (انتقل إلى خلية التحقق 17)"
elif _alive():
    status = "يعمل الآن"
elif pts:
    status = "متوقف — العملية غير حية ولا يوجد نموذج نهائي (راجع آخر السجل)"
else:
    status = "لم يبدأ بعد"

steps = pts[-1][0] if pts else 0
start = PIDF.stat().st_mtime if PIDF.exists() else (LOG.stat().st_mtime if LOG.exists() else time.time())
elapsed = max(time.time() - start, 1.0)
rate = steps / elapsed if steps else 0.0
eta_min = (MAXS - steps) / rate / 60 if rate > 0 else float("nan")
pct = min(100.0 * steps / MAXS, 100.0) if MAXS else 0.0
filled = int(30 * pct / 100)
bar = "#" * filled + "-" * (30 - filled)

print(f"الحالة: {status}")
print(f"[{bar}] {pct:5.1f}%   step {steps}/{MAXS}   معدل {rate:.2f} خطوة/ث   زمن منقضٍ {elapsed/60:.0f} د   متبقٍ ≈ {eta_min:.0f} د")
if pts:
    print(f"آخر خسارة (EMA): {pts[-1][1]:.4f} عند الخطوة {steps}")

# ---- حارس NaN / Inf (انهيار التدريب المبكر أشهر فشل صامت) ----
if pts and any((p[1] != p[1]) or float("inf") == abs(p[1]) for p in pts[-20:]):
    print("\n" + "!" * 62)
    print("تحذير: الخسارة انهارت إلى NaN/Inf!")
    print("الإجراء: أوقف التدريب (احذف train.pid وأعد الخلية 14) — سيبدأ من آخر نقطة سليمة.")
    print("إن تكرر: خفّض BATCH إلى النصف في الخلية 1.")
    print("!" * 62)

# ---- منحنى الخسارة ----
if len(pts) >= 2:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    xs, ys = zip(*pts)
    fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
    ax.plot(xs, ys, lw=1.3, color="#d62728")
    ax.set_xlabel("step")
    ax.set_ylabel("EMA loss")
    ax.set_title(f"loss curve - {CFG['RUN_NAME']} ({len(pts)} points)")
    ax.grid(alpha=0.3)
    chart = RUN_DIR / "loss_curve.png"
    fig.savefig(chart, dpi=110)
    plt.close(fig)
    display(IPyImage(filename=str(chart)))
    print(f"(حُفظ أيضاً بشكل دائم: {chart})")

# ---- آخر السجل ----
print("\n— آخر 12 سطراً من train_console.log —")
for ln in lines[-12:]:
    print("  " + ln)
ckpts = sorted(RUN_DIR.glob("DiT_epoch_*_step_*.pth"), key=lambda p: int(p.stem.rsplit("_", 1)[-1]))
if ckpts:
    print(f"\nنقاط التفتيش المحفوظة (آخر 3): " + ", ".join(c.name for c in ckpts[-3:]))


In [ ]:
# ============================================================ #
#  الخلية ١٦ — (اختيارية) انتظار اكتمال التدريب                #
#  مراقبة حية تلقائية كل 30 ثانية حتى النهاية                  #
#  (بدلها: شغّل الخلية 15 يدوياً كل فترة — نفس الغرض)           #
# ============================================================ #
import re, os, time
from pathlib import Path
from IPython.display import clear_output

LOG = RUN_DIR / "train_console.log"
PIDF = RUN_DIR / "train.pid"
FT = RUN_DIR / "ft_model.pth"
STEP_RE = re.compile(r"epoch (\d+), step (\d+), loss: ([0-9.eEnaif+-]+)")
MAXS = int(CFG["STEPS"])

def _alive():
    try:
        if PIDF.exists():
            os.kill(int(PIDF.read_text().strip()), 0)
            return True
    except Exception:
        pass
    return False

deadline = time.time() + 24 * 3600   # مهلة قصوى 24 ساعة
while time.time() < deadline:
    clear_output(wait=True)
    lines = LOG.read_text(errors="replace").splitlines() if LOG.exists() else []
    steps, loss = 0, None
    for ln in lines:
        m = STEP_RE.search(ln)
        if m:
            try:
                steps, loss = int(m.group(2)), float(m.group(3))
            except ValueError:
                pass
    pct = min(100.0 * steps / MAXS, 100.0)
    bar = "#" * int(30 * pct / 100)
    msg = f"[{bar:<30}] {pct:5.1f}%   step {steps}/{MAXS}"
    if loss is not None:
        msg += f"   loss {loss:.4f}"
        if loss != loss or loss == float("inf"):
            msg += "   << NaN/Inf — أوقف التدريب واستأنف من آخر نقطة!"
    print(msg, flush=True)
    if FT.exists():
        print("\nاكتمل التدريب — ft_model.pth جاهز. انتقل إلى خلية التحقق (17).")
        break
    if not _alive():
        print("\nتوقفت عملية التدريب دون إنتاج النموذج النهائي.")
        print("شغّل خلية المراقبة (15) لرؤية آخر السجل — غالباً يُستأنف بإعادة الخلية 14.")
        break
    time.sleep(30)
else:
    print("انتهت مهلة الانتظار (24 ساعة) — استخدم الخلية 15 للفحص اليدوي.")


<div dir="rtl">

## مرحلة التحقق والتقييم — بالأرقام لا بالانطباعات

قبل الاحتفال بالنموذج نجري ثلاثة فحوص متدرجة:

| الخلية | ماذا تفحص | المعيار |
|---|---|---|
| 17 — دخان أولي | أن النقطة المضبوطة **تعمل فعلاً** وتنتج صوتاً محوّلاً بجهازك | ملف صوتي يُنتج بلا أخطاء |
| 18 — **التقييم A/B الذهبي** | هل تحسّن المضبوط على الأساس **فعلاً**؟ تشابه متحدث أعلى بلا فقد محتوى — على مقاطع عراقية **لم ترها التدريب** | صعود متوسط تشابه CAMPPlus مع بقاء تشابه XLSR |
| 19 — **بنشمارك اللحظية** | كم يستغرق التحويل فعلياً على كرتك؟ زمن كتلة البث p50/p95 + RTF + الزمن الشامل لبروفايل الإنتاج `balanced` | RTF أقل بكثير من 1.0 وزمن كتلة داخل ميزانية 430ms |

**كيف يقرأ غير المختص نتائج A/B؟**
- **تشابه المتحدث (CAMPPlus)**: كم يشبه الصوت الخارجُ الصوتَ المرجعي العراقي (1.0 = تطابق تام). ارتفاعه بعد الضبط = النموذج صار أمهر بخصائص الحناجر العراقية.
- **الحفاظ على المحتوى (XLSR)**: هل بقيت الكلمات والمقاطع سليمة (1.0 = محتوى محفوظ تماماً). ثباته أو تحسنه = الضبط لم «يأكل» الوضوح.
- المقارنة عادلة: النموذجان يحوّلان **نفس المصدر إلى نفس المرجع بنفس البذرة العشوائية ونفس عدد خطوات التدفق**.

</div>


In [ ]:
# ============================================================ #
#  الخلية ١٧ — التحقق الأولي: استدلال حقيقي بالنقطة المضبوطة    #
#  (مصدر من أمثلة Seed-VC + مرجع عراقي من بياناتك -> تشغيل)    #
#  فشل هذه الخلية غير قاتل: الأوزان تصدَّر على أي حال           #
# ============================================================ #
from IPython.display import Audio, display
from pathlib import Path
import shutil

FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — شغّل خلية الإطلاق (14) وراقب حتى الاكتمال"
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

sh([VENV_PY, PIPE, "verify",
    "--repo", SEEDVC,
    "--run-dir", RUN_DIR,
    "--ckpt", FT,
    "--config", CFG_TINY,
    "--data-dir", DATA_DIR], check=False)   # فشل verify ليس قاتلاً — الأوزان تصدَّر على أي حال

outs = sorted((RUN_DIR / "verify_out").glob("vc_*.wav"))
if not outs:
    # احتياط (جديد): verify يعتمد على مجلد yodas — إن لم يوجد ننفذ استدلالاً مباشراً
    # بمرجع من بيانات الاختبار المعزولة أو من أي بيانات متاحة
    ref = None
    for cand_dir in (TEST_DIR, DATA_DIR / "yodas", DATA_DIR / "youssef", DATA_DIR / "safaa"):
        if cand_dir.exists():
            cands = sorted(cand_dir.glob("*.ogg"))[:1] or sorted(cand_dir.glob("*.wav"))[:1]
            if cands:
                ref = cands[0]
                break
    if ref is not None:
        print(f"\n[احتياط] استدلال مباشر بمرجع: {ref.name}")
        outdir = RUN_DIR / "verify_out"
        outdir.mkdir(parents=True, exist_ok=True)
        sh([VENV_PY, "inference.py",
            "--checkpoint", FT,
            "--config", CFG_TINY,
            "--source", SEEDVC / "examples" / "source" / "source_s1.wav",
            "--target", ref,
            "--output", outdir,
            "--diffusion-steps", "16", "--fp16", "true"],
           cwd=SEEDVC, check=False)
        outs = sorted(outdir.glob("vc_*.wav"))

if outs:
    print(f"\nناتج التحويل: {outs[0].name}  ({outs[0].stat().st_size/1e6:.1f} MB)")
    print("استمع للناتج (المصدر إنجليزي يحمل الطابع الصوتي العراقي المرجعي):")
    display(Audio(filename=str(outs[0])))
    for extra in outs[1:3]:
        print(f"ناتج إضافي: {extra.name}")
        display(Audio(filename=str(extra)))
    print(f"\n[نجاح] النقطة المضبوطة تعمل على جهازك — انتقل إلى التقييم الكمي (الخلية 18).")
else:
    print("لم يُنتج ملف صوتي — راجع السجل أعلاه. (غير قاتل: الأوزان ستصدَّر في الخلية 20 على أي حال)")


In [ ]:
# ============================================================ #
#  الخلية ١٨ — التقييم A/B الذهبي: الأساس مقابل المضبوط        #
#  (جديد في الإصدار ٢ — قلب ادعاء «الأفضل» بالأرقام)           #
#                                                              #
#  نكتب مكتبة تقييم مشتركة + سكربت A/B ثم نشغّلهما:            #
#   • نفس أزواج الاختبار العراقية (معزولة عن التدريب)          #
#   • نفس البذرة العشوائية ونفس خطوات التدفق للنموذجين         #
#   • مقاييس موضوعية: تشابه متحدث CAMPPlus + حفظ محتوى XLSR    #
# ============================================================ #
import json
from pathlib import Path

# ---------- المكتبة المشتركة (يستوردها سكربتا A/B والبنشمارك) ----------
LIB = ROOT / "rafidain_eval_lib.py"
LIB.write_text(r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Shared evaluation utilities for the Rafidain RunPod notebook (v2).

Faithful ports, no reinvention:
  - stack loading      -> seed-vc inference.py load_models()
  - utterance convert  -> seed-vc inference.py main() (non-f0 path)
  - streaming session  -> rafidain infrastructure/seedvc/engine.py
                          (itself a port of real-time-gui.py)

Run with cwd = seed-vc repository root.
"""
import os, sys, math, argparse
import numpy as np
import torch
import torchaudio
import librosa

SEEDVC_ROOT = os.getcwd()
if SEEDVC_ROOT not in sys.path:
    sys.path.insert(0, SEEDVC_ROOT)

MODEL_FRAMES_PER_SECOND = 50   # seed-vc works on 20 ms frames
SAMPLE_RATE_IN = 16_000
CE_DIT_DIFFERENCE_S = 2.0      # XLSR vs DiT frame offset (upstream constant)


# ------------------------------------------------------------------ loading
def load_stack(checkpoint, config, fp16=True):
    """Load the tiny-XLSR stack exactly like inference.py does."""
    from inference import load_models
    args = argparse.Namespace(checkpoint=str(checkpoint), config=str(config),
                              fp16=bool(fp16), f0_condition=False)
    return load_models(args)


def find_base_ckpt():
    """Locate (or fetch) the published base checkpoint DiT_uvit_tat_xlsr_ema.pth."""
    from hf_utils import load_custom_model_from_hf
    return load_custom_model_from_hf("Plachta/Seed-VC", "DiT_uvit_tat_xlsr_ema.pth", None)


def swap_weights(model, ckpt_path):
    """Hot-swap DiT weights in-place (base -> fine-tuned or back)."""
    from modules.commons import load_checkpoint
    model, _, _, _ = load_checkpoint(model, None, str(ckpt_path),
                                     load_only_params=True, ignore_modules=[],
                                     is_distributed=False)
    for key in model:
        model[key].eval()
    model.cfm.estimator.setup_caches(max_batch_size=1, max_seq_length=8192)
    return model


# ------------------------------------------------------ utterance conversion
@torch.no_grad()
def convert_utterance(model, semantic_fn, vocoder_fn, campplus_model, to_mel,
                      src_path, ref_path, sr, diffusion_steps=25,
                      length_adjust=1.0, inference_cfg_rate=0.7, fp16=True,
                      max_src_sec=None):
    """Port of inference.py main() non-f0 path. Returns (tensor (1,T), out_sr)."""
    from inference import crossfade
    device = next(model.cfm.parameters()).device

    source_audio = librosa.load(src_path, sr=sr)[0]
    ref_audio = librosa.load(ref_path, sr=sr)[0]
    if max_src_sec:
        source_audio = source_audio[: int(sr * max_src_sec)]
    source_audio = torch.tensor(source_audio).unsqueeze(0).float().to(device)
    ref_audio = torch.tensor(ref_audio[: sr * 25]).unsqueeze(0).float().to(device)

    hop_length = 256
    max_context_window = sr // hop_length * 30
    overlap_frame_len = 16
    overlap_wave_len = overlap_frame_len * hop_length

    converted_waves_16k = torchaudio.functional.resample(source_audio, sr, 16000)
    S_alt = semantic_fn(converted_waves_16k)
    ori_waves_16k = torchaudio.functional.resample(ref_audio, sr, 16000)
    S_ori = semantic_fn(ori_waves_16k)

    mel = to_mel(source_audio.to(device).float())
    mel2 = to_mel(ref_audio.to(device).float())

    target_lengths = torch.LongTensor([int(mel.size(2) * length_adjust)]).to(mel.device)
    target2_lengths = torch.LongTensor([mel2.size(2)]).to(mel2.device)

    feat2 = torchaudio.compliance.kaldi.fbank(ori_waves_16k, num_mel_bins=80,
                                              dither=0, sample_frequency=16000)
    feat2 = feat2 - feat2.mean(dim=0, keepdim=True)
    style2 = campplus_model(feat2.unsqueeze(0))

    cond, _, _, _, _ = model.length_regulator(S_alt, ylens=target_lengths,
                                              n_quantizers=3, f0=None)
    prompt_condition, _, _, _, _ = model.length_regulator(S_ori, ylens=target2_lengths,
                                                          n_quantizers=3, f0=None)

    max_source_window = max_context_window - mel2.size(2)
    processed_frames = 0
    generated_wave_chunks = []
    while processed_frames < cond.size(1):
        chunk_cond = cond[:, processed_frames: processed_frames + max_source_window]
        is_last_chunk = processed_frames + max_source_window >= cond.size(1)
        cat_condition = torch.cat([prompt_condition, chunk_cond], dim=1)
        with torch.autocast(device_type=device.type,
                            dtype=torch.float16 if fp16 else torch.float32):
            vc_target = model.cfm.inference(cat_condition,
                                            torch.LongTensor([cat_condition.size(1)]).to(mel2.device),
                                            mel2, style2, None, diffusion_steps,
                                            inference_cfg_rate=inference_cfg_rate)
            vc_target = vc_target[:, :, mel2.size(-1):]
        vc_wave = vocoder_fn(vc_target.float()).squeeze()
        vc_wave = vc_wave[None, :]
        if processed_frames == 0:
            if is_last_chunk:
                generated_wave_chunks.append(vc_wave[0].cpu().numpy())
                break
            generated_wave_chunks.append(vc_wave[0, :-overlap_wave_len].cpu().numpy())
            previous_chunk = vc_wave[0, -overlap_wave_len:]
            processed_frames += vc_target.size(2) - overlap_frame_len
        elif is_last_chunk:
            generated_wave_chunks.append(
                crossfade(previous_chunk.cpu().numpy(), vc_wave[0].cpu().numpy(), overlap_wave_len))
            processed_frames += vc_target.size(2) - overlap_frame_len
            break
        else:
            generated_wave_chunks.append(
                crossfade(previous_chunk.cpu().numpy(),
                          vc_wave[0, :-overlap_wave_len].cpu().numpy(), overlap_wave_len))
            previous_chunk = vc_wave[0, -overlap_wave_len:]
            processed_frames += vc_target.size(2) - overlap_frame_len
    vc_wave = torch.tensor(np.concatenate(generated_wave_chunks))[None, :].float()
    return vc_wave, sr


# ------------------------------------------------------------------ metrics
def _campplus_embed(campplus_model, wav_np_16k):
    t = torch.from_numpy(np.asarray(wav_np_16k, dtype=np.float32).reshape(1, -1))
    feat = torchaudio.compliance.kaldi.fbank(t, num_mel_bins=80, dither=0,
                                             sample_frequency=16000)
    feat = feat - feat.mean(dim=0, keepdim=True)
    return campplus_model(feat.unsqueeze(0)).reshape(-1)


def speaker_similarity(campplus_model, wav_a_16k, wav_b_16k):
    """CAMPPlus cosine — timbre closeness to the reference speaker."""
    a = _campplus_embed(campplus_model, wav_a_16k)
    b = _campplus_embed(campplus_model, wav_b_16k)
    return float(torch.nn.functional.cosine_similarity(a, b, dim=0))


def _xlsr_embed(semantic_fn, wav_np_16k):
    t = torch.from_numpy(np.asarray(wav_np_16k, dtype=np.float32).reshape(1, -1))
    with torch.no_grad():
        s = semantic_fn(t)
    return s.reshape(-1, s.size(-1)).mean(dim=0)


def content_similarity(semantic_fn, wav_a_16k, wav_b_16k):
    """XLSR mean-pooled cosine — did the words survive the conversion?"""
    a = _xlsr_embed(semantic_fn, wav_a_16k)
    b = _xlsr_embed(semantic_fn, wav_b_16k)
    return float(torch.nn.functional.cosine_similarity(a, b, dim=0))


def load_16k(path, max_sec=None):
    y = librosa.load(path, sr=16000)[0]
    return y[: int(16000 * max_sec)] if max_sec else y


# ------------------------------------------------- streaming session (port)
class StreamBenchSession:
    """Port of rafidain infrastructure/seedvc/engine.py SeedVCConverterSession.

    Identical per-block mathematics to the production engine (which itself
    ports real-time-gui.py): sliding window, XLSR content features, length
    regulation, CFM at `diffusion_steps`, HiFT vocoder, equal-power crossfade.
    """

    def __init__(self, stack, profile, ref_path):
        (model, semantic_fn, f0_fn, vocoder_fn, campplus_model, to_mel, mel_fn_args) = stack
        self.model = model
        self.semantic_fn = semantic_fn
        self.vocoder_fn = vocoder_fn
        self.campplus = campplus_model
        self.to_mel = to_mel
        self.kaldi = torchaudio.compliance.kaldi
        self.sr = int(mel_fn_args["sampling_rate"])
        self.hop_length = int(mel_fn_args["hop_size"])
        self.device = next(model.cfm.parameters()).device
        self.fp16 = self.device.type == "cuda"
        self.resample_up = torchaudio.transforms.Resample(16_000, self.sr).to(self.device)
        self.profile = profile

        sr, zc = self.sr, self.sr // MODEL_FRAMES_PER_SECOND

        def q(ms):
            return int(round(ms / 1000.0 * sr / zc)) * zc

        self.block_frame = q(profile["block_ms"])
        self.block_frame_16k = 320 * self.block_frame // zc
        self.crossfade_frame = q(profile["crossfade_ms"])
        self.extra_frame = q(profile["context_left_ms"])
        self.extra_frame_right = q(profile["context_right_ms"])
        self.skip_head = self.extra_frame // zc
        self.skip_tail = self.extra_frame_right // zc
        self.return_length = (self.block_frame + self.crossfade_frame + zc) // zc
        self.ce_dit_frames = int(CE_DIT_DIFFERENCE_S * MODEL_FRAMES_PER_SECOND)

        window = (self.extra_frame + self.crossfade_frame + zc + self.block_frame
                  + self.extra_frame_right)
        self.input_wav = torch.zeros(window, device=self.device, dtype=torch.float32)
        self.input_wav_res = torch.zeros(320 * window // zc, device=self.device,
                                         dtype=torch.float32)

        self.out_len = self.return_length * sr // MODEL_FRAMES_PER_SECOND
        self.overlap = self.out_len - self.block_frame
        assert self.overlap > 0 and self.block_frame > self.overlap, "block_ms too small"
        ramp = torch.sin(0.5 * float(np.pi) * torch.linspace(
            0.0, 1.0, steps=self.overlap, device=self.device)) ** 2
        self.fade_in = ramp.float()
        self._fade_in_np = ramp.detach().cpu().numpy().astype(np.float32)
        self._fade_out_np = 1.0 - self._fade_in_np
        self._carry = None
        self._pending = np.zeros(0, dtype=np.float32)

        # ---- condition on the reference voice (same as production engine) ----
        max_ref = int(16000 * profile["max_prompt_s"])
        ref16 = load_16k(ref_path)[:max_ref]
        assert ref16.size >= 16000, "reference too short"
        ref16_t = torch.from_numpy(ref16).to(self.device)
        ref_sr_t = self.resample_up(ref16_t.reshape(1, -1)).reshape(-1)

        with torch.no_grad():
            s_ori = self.semantic_fn(ref16_t.reshape(1, -1))
            feat = self.kaldi.fbank(ref16_t.reshape(1, -1), num_mel_bins=80,
                                    dither=0, sample_frequency=SAMPLE_RATE_IN)
            feat = feat - feat.mean(dim=0, keepdim=True)
            self.style2 = self.campplus(feat.unsqueeze(0))
            self.mel2 = self.to_mel(ref_sr_t.reshape(1, -1))
            target2_lengths = torch.LongTensor([self.mel2.size(2)]).to(self.device)
            self.prompt_condition = self.model.length_regulator(
                s_ori, ylens=target2_lengths, n_quantizers=3, f0=None)[0]

    def convert_block(self, pcm):
        self._pending = np.concatenate(
            [self._pending, np.asarray(pcm, dtype=np.float32).reshape(-1)])
        outs = []
        while self._pending.size >= self.block_frame_16k:
            block = self._pending[: self.block_frame_16k]
            self._pending = self._pending[self.block_frame_16k:]
            outs.append(self._process_one(block))
        if not outs:
            return np.zeros(0, dtype=np.float32)
        return np.concatenate(outs) if len(outs) > 1 else outs[0]

    def finish(self):
        tail = np.zeros(0, dtype=np.float32)
        if self._pending.size:
            padded = np.pad(self._pending, (0, self.block_frame_16k - self._pending.size))
            self._pending = np.zeros(0, dtype=np.float32)
            tail = self._process_one(padded, final=True)
        if self._carry is not None:
            carry = self._carry.detach().cpu().numpy()
            self._carry = None
            tail = np.concatenate([tail, carry]) if tail.size else carry
        return tail

    def _process_one(self, block16k, final=False):
        device = self.device
        block16k_t = torch.from_numpy(np.ascontiguousarray(block16k)).to(device)
        with torch.no_grad():
            self.input_wav = torch.cat(
                [self.input_wav[self.block_frame:], self._up(block16k_t)]
            )[: self.input_wav.shape[0]]
            self.input_wav_res = torch.cat(
                [self.input_wav_res[self.block_frame_16k:], block16k_t]
            )[: self.input_wav_res.shape[0]]

            s_alt = self.semantic_fn(self.input_wav_res.reshape(1, -1))
            s_alt = s_alt[:, self.ce_dit_frames:]
            target_lengths = torch.LongTensor(
                [int((self.skip_head + self.return_length + self.skip_tail
                      - self.ce_dit_frames) / MODEL_FRAMES_PER_SECOND
                     * self.sr // self.hop_length)]).to(device)
            cond = self.model.length_regulator(s_alt, ylens=target_lengths,
                                               n_quantizers=3, f0=None)[0]
            cat_condition = torch.cat([self.prompt_condition, cond], dim=1)

            if self.fp16:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    vc_target = self.model.cfm.inference(
                        cat_condition,
                        torch.LongTensor([cat_condition.size(1)]).to(device),
                        self.mel2, self.style2, None,
                        self.profile["diffusion_steps"],
                        inference_cfg_rate=self.profile["inference_cfg_rate"])
            else:
                vc_target = self.model.cfm.inference(
                    cat_condition,
                    torch.LongTensor([cat_condition.size(1)]).to(device),
                    self.mel2, self.style2, None,
                    self.profile["diffusion_steps"],
                    inference_cfg_rate=self.profile["inference_cfg_rate"])
            vc_target = vc_target[:, :, self.mel2.size(-1):]
            vc_wave = self.vocoder_fn(vc_target.float()).squeeze()

            output_len = self.return_length * self.sr // MODEL_FRAMES_PER_SECOND
            tail_len = self.skip_tail * self.sr // MODEL_FRAMES_PER_SECOND
            wave = vc_wave.detach().float().cpu().numpy().reshape(-1)
            need = output_len + tail_len
            if wave.size < need:
                wave = np.pad(wave, (0, need - wave.size))
            out = (wave[-need:-tail_len] if tail_len > 0 else wave[-output_len:]).astype(np.float32)

            ov = self.overlap
            if self._carry is not None:
                carry = self._carry.detach().cpu().numpy().astype(np.float32)
                head = out[:ov] * self._fade_in_np + carry[:ov] * self._fade_out_np
                body = out[ov:] if final else out[ov: self.out_len - ov]
                emitted = np.concatenate([head, body]).astype(np.float32)
                self._carry = None if final else torch.from_numpy(out[-ov:].copy()).to(device)
            else:
                if final:
                    emitted = out.astype(np.float32)
                    self._carry = None
                else:
                    emitted = out[: self.out_len - ov].astype(np.float32)
                    self._carry = torch.from_numpy(out[-ov:].copy()).to(device)
        return emitted

    def _up(self, block16k_t):
        up = self.resample_up(block16k_t.reshape(1, -1)).reshape(-1)
        if up.shape[0] > self.block_frame:
            up = up[: self.block_frame]
        elif up.shape[0] < self.block_frame:
            up = torch.nn.functional.pad(up, (0, self.block_frame - up.shape[0]))
        return up
''', encoding="utf-8")
print(f"[مكتبة التقييم] كُتبت: {LIB}")

# ---------- سكربت A/B: الأساس مقابل المضبوط ----------
AB = ROOT / "ab_eval.py"
AB.write_text(r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""A/B evaluation: published base Seed-VC vs Rafidain Iraqi fine-tune.

Fair by construction: identical held-out Iraqi pairs, identical random seed
per pair (same CFM noise), identical diffusion steps. Reports CAMPPlus
speaker similarity + XLSR content preservation per pair and averaged.
"""
import os, sys, json, glob, time, argparse, random
os.environ.setdefault("HF_HUB_CACHE", "./checkpoints/hf_cache")
sys.path.insert(0, os.environ.get("NB_ROOT", "."))

import numpy as np
import torch
import torchaudio
from rafidain_eval_lib import (load_stack, find_base_ckpt, swap_weights,
                               convert_utterance, speaker_similarity,
                               content_similarity, load_16k)

AUDIO_EXTS = (".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus")


def list_audio(d):
    out = []
    for root, _, files in os.walk(d):
        for f in sorted(files):
            if f.lower().endswith(AUDIO_EXTS):
                out.append(os.path.join(root, f))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ft-ckpt", required=True)
    ap.add_argument("--config", required=True)
    ap.add_argument("--test-dir", required=True)
    ap.add_argument("--fallback-dir", default=None)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--pairs", type=int, default=5)
    ap.add_argument("--diffusion-steps", type=int, default=25)
    args = ap.parse_args()

    random.seed(7)
    np.random.seed(7)
    torch.manual_seed(7)

    clips = list_audio(args.test_dir)
    in_train = False
    if len(clips) < 2 * args.pairs and args.fallback_dir:
        extra = [p for p in list_audio(args.fallback_dir)
                 if os.path.basename(p) not in {os.path.basename(c) for c in clips}]
        clips = clips + extra
        in_train = True
    assert len(clips) >= 2 * args.pairs, (
        f"need {2*args.pairs} clips, found {len(clips)}")
    if in_train:
        print("[WARN] test set too small - falling back to TRAINING clips "
              "(evaluation less honest; prefer the holdout cell)")

    random.shuffle(clips)
    sources = clips[: args.pairs]
    refs = clips[args.pairs: 2 * args.pairs]

    os.makedirs(args.out_dir, exist_ok=True)

    print("[load] base stack (DiT_uvit_tat_xlsr_ema) ...")
    base_ckpt = find_base_ckpt()
    t0 = time.time()
    stack = load_stack(base_ckpt, args.config, fp16=True)
    model, semantic_fn, f0_fn, vocoder_fn, campplus_model, to_mel, mel_fn_args = stack
    sr = int(mel_fn_args["sampling_rate"])
    print(f"[load] ready in {time.time()-t0:.0f}s (sr={sr})")

    results = {"config": {"diffusion_steps": args.diffusion_steps,
                          "pairs": args.pairs, "sr": sr,
                          "held_out": not in_train},
               "pairs": [], "summary": {}}

    def eval_model(tag):
        rows, fails = [], []
        for i, (src, ref) in enumerate(zip(sources, refs)):
            try:
                torch.manual_seed(1000 + i)      # same CFM noise for both models
                wav, out_sr = convert_utterance(
                    model, semantic_fn, vocoder_fn, campplus_model, to_mel,
                    src, ref, sr, diffusion_steps=args.diffusion_steps,
                    fp16=True, max_src_sec=12.0)
                out_name = f"{tag}_pair{i:02d}.wav"
                torchaudio.save(os.path.join(args.out_dir, out_name), wav.cpu(), out_sr)
                conv16 = wav.reshape(-1).cpu().numpy()
                conv16 = torchaudio.functional.resample(
                    torch.from_numpy(conv16), out_sr, 16000).numpy()
                sim_spk = speaker_similarity(campplus_model, conv16, load_16k(ref, 25))
                sim_cnt = content_similarity(semantic_fn, conv16, load_16k(src, 12))
                rows.append({"pair": i, "source": os.path.basename(src),
                             "reference": os.path.basename(ref),
                             "speaker_sim": round(sim_spk, 4),
                             "content_sim": round(sim_cnt, 4)})
                print(f"  [{tag}] pair {i}: speaker={sim_spk:.4f}  content={sim_cnt:.4f}")
            except Exception as exc:   # مقطع تالف لا يوقف التقييم كله
                fails.append({"pair": i, "error": str(exc)[:200]})
                print(f"  [{tag}] pair {i}: FAILED ({exc})")
        if fails:
            results.setdefault("failures", {})[tag] = fails
        assert rows, f"all pairs failed for {tag} — check audio files / logs above"
        return rows

    print("\n[eval] BASE model ...")
    results["base"] = eval_model("base")

    print("\n[swap] loading fine-tuned weights ...")
    swap_weights(model, args.ft_ckpt)

    print("\n[eval] FINE-TUNED model ...")
    results["ft"] = eval_model("ft")

    # ---- aggregate (على الأزواج المشتركة الناجحة في النموذجين — مقارنة عادلة) ----
    common = sorted({r["pair"] for r in results["base"]} &
                    {r["pair"] for r in results["ft"]})
    if len(common) < len(sources):
        print(f"[note] computing means over {len(common)} common successful pairs")
    b_spk = [r["speaker_sim"] for r in results["base"] if r["pair"] in common]
    f_spk = [r["speaker_sim"] for r in results["ft"] if r["pair"] in common]
    b_cnt = [r["content_sim"] for r in results["base"] if r["pair"] in common]
    f_cnt = [r["content_sim"] for r in results["ft"] if r["pair"] in common]
    base_spk = float(np.mean(b_spk))
    ft_spk = float(np.mean(f_spk))
    base_cnt = float(np.mean(b_cnt))
    ft_cnt = float(np.mean(f_cnt))
    results["summary"] = {
        "base_speaker_sim_mean": round(base_spk, 4),
        "ft_speaker_sim_mean": round(ft_spk, 4),
        "speaker_sim_delta": round(ft_spk - base_spk, 4),
        "base_content_sim_mean": round(base_cnt, 4),
        "ft_content_sim_mean": round(ft_cnt, 4),
        "content_sim_delta": round(ft_cnt - base_cnt, 4),
    }
    s = results["summary"]
    verdict = "IMPROVED" if (s["speaker_sim_delta"] > 0
                             and s["content_sim_delta"] > -0.02) else "NEUTRAL/REGRESSION"
    results["summary"]["verdict"] = verdict

    out_json = os.path.join(args.out_dir, "ab_eval.json")
    with open(out_json, "w", encoding="utf-8") as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2)

    print("\n" + "=" * 70)
    print("A/B SUMMARY  (CAMPPlus speaker sim | XLSR content sim)")
    print("=" * 70)
    print(f"  base model : {base_spk:.4f} | {base_cnt:.4f}")
    print(f"  fine-tuned : {ft_spk:.4f} | {ft_cnt:.4f}")
    print(f"  delta      : {s['speaker_sim_delta']:+.4f} | {s['content_sim_delta']:+.4f}")
    print(f"  verdict    : {verdict}")
    print(f"  (details)  : {out_json}")
    print("=" * 70)


if __name__ == "__main__":
    main()
''', encoding="utf-8")
print(f"[سكربت A/B] كُتب: {AB}")

# ---------- تشغيل التقييم ----------
FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — أكمل التدريب أولاً (الخلية 14)"
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"
AB_DIR = RUN_DIR / "ab_eval"

env = dict(ENV)
env["NB_ROOT"] = str(ROOT)
sh([VENV_PY, AB,
    "--ft-ckpt", FT,
    "--config", CFG_TINY,
    "--test-dir", TEST_DIR,
    "--fallback-dir", DATA_DIR,
    "--out-dir", AB_DIR,
    "--pairs", int(CFG["EVAL_PAIRS"]),
    "--diffusion-steps", int(CFG["EVAL_STEPS"])],
   cwd=SEEDVC, env=env)

# ---------- عرض النتائج: جدول + استماع مقارن ----------
from IPython.display import Audio, display

rep = json.loads((AB_DIR / "ab_eval.json").read_text(encoding="utf-8"))
s = rep["summary"]
print(f"""
تشابه المتحدث (CAMPPlus):  الأساس {s['base_speaker_sim_mean']:.4f}  ←  المضبوط {s['ft_speaker_sim_mean']:.4f}   (فرق {s['speaker_sim_delta']:+.4f})
الحفاظ على المحتوى (XLSR):  الأساس {s['base_content_sim_mean']:.4f}  ←  المضبوط {s['ft_content_sim_mean']:.4f}   (فرق {s['content_sim_delta']:+.4f})
الفيصل: {s['verdict']}
{'⚠ المقاطع من بيانات التدريب (لا يوجد عزل) — النتيجة متفائلة نسبياً؛ شغّل خلية العزل 12 وأعد التقييم' if not rep['config']['held_out'] else '✔ التقييم على مقاطع معزولة لم يدخلها التدريب'}
""")
print("استمع للمقارنة (الزوج الأول): المصدر ← الأساس ← المضبوط، مع المرجع العراقي:")
pair0_src = next((p["source"] for p in rep["ft"]), None)
if pair0_src:
    print(f"- المصدر: {pair0_src}")
for tag, label in (("base", "ناتج الأساس"), ("ft", "ناتج المضبوط")):
    f = AB_DIR / f"{tag}_pair00.wav"
    if f.exists():
        print(f"- {label}:")
        display(Audio(filename=str(f)))
ref_name = rep["ft"][0]["reference"] if rep["ft"] else None
ref_f = None
if ref_name:
    for d in (TEST_DIR, DATA_DIR / "yodas", DATA_DIR / "youssef", DATA_DIR / "safaa", DATA_DIR / "custom"):
        cand = d / ref_name
        if cand.exists():
            ref_f = cand
            break
if ref_f:
    print(f"- المرجع العراقي: {ref_f.name}")
    display(Audio(filename=str(ref_f)))


In [ ]:
# ============================================================ #
#  الخلية ١٩ — بنشمارك الاستنساخ اللحظي (جديد في الإصدار ٢)     #
#                                                              #
#  يقيس على كرتك الفعلي، بالنموذج المضبوط:                      #
#   أ) RTF على مستوى المقطع عند 10/16/30 خطوة تدفق             #
#   ب) **البث الإنتاجي الحقيقي**: بروفايل balanced               #
#      (كتل 180ms + crossfade 40ms + سياق 20ms + 10 خطوات)     #
#      بنفس منطج محرك رافِدَين الإنتاجي — يعطي p50/p95          #
#      وزمن الوصول الشامل المقدَّر                              #
# ============================================================ #
import json
from pathlib import Path

BENCH = ROOT / "bench_stream.py"
BENCH.write_text(r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Latency benchmark for the fine-tuned Iraqi model.

Part A: utterance-level RTF at diffusion steps {10, 16, 30}.
Part B: production streaming simulation with the `balanced` profile
        (block 180ms / crossfade 40ms / right ctx 20ms / 10 steps) using
        StreamBenchSession - a faithful port of rafidain's production
        engine (which ports seed-vc's real-time-gui.py).

RTF < 1.0 means faster than real time. The published measurement for this
model family on the balanced profile is ~430 ms end-to-end on a decent GPU.
"""
import os, sys, json, time, argparse, random
os.environ.setdefault("HF_HUB_CACHE", "./checkpoints/hf_cache")
sys.path.insert(0, os.environ.get("NB_ROOT", "."))

import numpy as np
import torch
import torchaudio
from rafidain_eval_lib import load_stack, convert_utterance, load_16k, StreamBenchSession

AUDIO_EXTS = (".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus")


def list_audio(d):
    out = []
    for root, _, files in os.walk(d):
        for f in sorted(files):
            if f.lower().endswith(AUDIO_EXTS):
                out.append(os.path.join(root, f))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ft-ckpt", required=True)
    ap.add_argument("--config", required=True)
    ap.add_argument("--test-dir", required=True)
    ap.add_argument("--fallback-dir", default=None)
    ap.add_argument("--out-dir", required=True)
    args = ap.parse_args()

    torch.manual_seed(7)
    random.seed(7)
    np.random.seed(7)

    clips = list_audio(args.test_dir)
    if len(clips) < 2 and args.fallback_dir:
        clips = clips + list_audio(args.fallback_dir)
    assert len(clips) >= 2, "need at least 2 clips (source + reference)"

    # جرب المقاطع بالترتيب حتى ينجح تحميل واحد (مقطع تالف لا يكسر البنشمارك)
    import librosa as _lib
    def _try_load(path, sr=16000, max_sec=None):
        try:
            y = _lib.load(path, sr=sr)[0]
            if max_sec:
                y = y[: int(sr * max_sec)]
            return y if y.size > sr else None   # تجاهل الأقصر من ثانية
        except Exception:
            return None

    src = ref = None
    for c in clips:
        if src is None and _try_load(c) is not None:
            src = c
        elif ref is None and _try_load(c) is not None:
            ref = c
        if src is not None and ref is not None:
            break
    assert src is not None and ref is not None, "no loadable clips found"
    print(f"[bench] source: {os.path.basename(src)}")
    print(f"[bench] reference: {os.path.basename(ref)}")

    print("\n[load] fine-tuned stack ...")
    t0 = time.time()
    stack = load_stack(args.ft_ckpt, args.config, fp16=True)
    model, semantic_fn, f0_fn, vocoder_fn, campplus_model, to_mel, mel_fn_args = stack
    sr = int(mel_fn_args["sampling_rate"])
    print(f"[load] ready in {time.time()-t0:.0f}s")

    os.makedirs(args.out_dir, exist_ok=True)
    report = {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
              "sr": sr, "utterance_rtf": {}, "streaming": {}}

    # ---- Part A: utterance RTF ----
    print("\n[A] utterance-level RTF (convert ~10s clip, model already loaded):")
    for steps in (10, 16, 30):
        # warmup once for kernel autotuning
        convert_utterance(model, semantic_fn, vocoder_fn, campplus_model, to_mel,
                          src, ref, sr, diffusion_steps=steps, fp16=True, max_src_sec=10)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t0 = time.perf_counter()
        wav, out_sr = convert_utterance(model, semantic_fn, vocoder_fn, campplus_model,
                                        to_mel, src, ref, sr, diffusion_steps=steps,
                                        fp16=True, max_src_sec=10)
        dur = wav.size(-1) / out_sr
        wall = time.perf_counter() - t0
        rtf = wall / dur
        report["utterance_rtf"][str(steps)] = {"wall_s": round(wall, 3),
                                               "audio_s": round(dur, 2),
                                               "rtf": round(rtf, 4)}
        print(f"    steps={steps:>2}: {wall:.2f}s wall for {dur:.1f}s audio  ->  RTF {rtf:.3f}"
              + ("  (أسرع من الزمن الحقيقي)" if rtf < 1 else "  (أبطأ من الزمن الحقيقي!)"))

    # ---- Part B: streaming with production profile ----
    profiles = {
        "balanced": dict(block_ms=180, crossfade_ms=40, context_left_ms=2500,
                         context_right_ms=20, diffusion_steps=10,
                         inference_cfg_rate=0.7, max_prompt_s=3.0),
        "ultra_low": dict(block_ms=120, crossfade_ms=40, context_left_ms=2000,
                          context_right_ms=20, diffusion_steps=4,
                          inference_cfg_rate=0.0, max_prompt_s=3.0),
    }
    src16 = load_16k(src, 8.0)   # 8 ثوانٍ من الكلام العراقي الحقيقي
    print("\n[B] streaming simulation (production engine math):")
    for name, prof in profiles.items():
        sess = StreamBenchSession(stack, prof, ref)
        block = sess.block_frame_16k
        times = []
        emitted_all = []
        pos = 0
        n_blocks = 0
        while pos + block <= src16.size:
            pcm = src16[pos: pos + block]
            t0 = time.perf_counter()
            out = sess.convert_block(pcm)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            emitted_all.append(out)
            if n_blocks >= 3:          # skip first blocks (CUDA warmup)
                times.append(dt)
            n_blocks += 1
            pos += block
        emitted_all.append(sess.finish())
        if not times:
            print(f"    [{name}] المصدر أقصر من 3 كتل — تخطّي")
            continue
        if name == "balanced" and emitted_all:
            # احتفظ بصوت البث للاستماع — دليل ملموس أن مسار الإنتاج يعمل بالنقطة المضبوطة
            import numpy as _np
            full = _np.concatenate([e for e in emitted_all if e.size])
            torchaudio.save(os.path.join(args.out_dir, "stream_balanced_out.wav"),
                            torch.from_numpy(full).reshape(1, -1).float(), sess.sr)
            print(f"    [{name}] الصوت المتدفق حُفظ: stream_balanced_out.wav "
                  f"({full.size/sess.sr:.1f}s)")
        times_np = np.array(times)
        p50 = float(np.percentile(times_np, 50)) * 1000
        p95 = float(np.percentile(times_np, 95)) * 1000
        mean = float(times_np.mean()) * 1000
        rtf = float(times_np.mean()) / (prof["block_ms"] / 1000)
        algo = prof["block_ms"] + prof["crossfade_ms"] + prof["context_right_ms"]
        e2e = algo + mean
        report["streaming"][name] = {
            "block_ms": prof["block_ms"], "diffusion_steps": prof["diffusion_steps"],
            "n_blocks_measured": int(times_np.size),
            "block_time_p50_ms": round(p50, 1), "block_time_p95_ms": round(p95, 1),
            "block_time_mean_ms": round(mean, 1),
            "algorithmic_delay_ms": algo,
            "estimated_e2e_ms": round(e2e, 1),
            "rtf": round(rtf, 3),
        }
        print(f"    [{name}] كتلة {prof['block_ms']}ms × {prof['diffusion_steps']} خطوة:")
        print(f"        زمن معالجة الكتلة: p50 {p50:.0f}ms | p95 {p95:.0f}ms | RTF {rtf:.2f}")
        print(f"        زمن الوصول الشامل المقدَّر (كتلة+crossfade+سياق+استدلال): {e2e:.0f}ms"
              + ("  ← داخل ميزانية المكالمات" if e2e < 500 else "  ← فوق الميزانية"))

    out_json = os.path.join(args.out_dir, "bench_stream.json")
    with open(out_json, "w", encoding="utf-8") as fh:
        json.dump(report, fh, ensure_ascii=False, indent=2)
    print(f"\n[bench] report: {out_json}")


if __name__ == "__main__":
    main()
''', encoding="utf-8")
print(f"[سكربت البنشمارك] كُتب: {BENCH}")

# ---------- تشغيل البنشمارك ----------
FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — أكمل التدريب أولاً (الخلية 14)"
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"
BENCH_DIR = RUN_DIR / "bench"

env = dict(ENV)
env["NB_ROOT"] = str(ROOT)
sh([VENV_PY, BENCH,
    "--ft-ckpt", FT,
    "--config", CFG_TINY,
    "--test-dir", TEST_DIR,
    "--fallback-dir", DATA_DIR,
    "--out-dir", BENCH_DIR],
   cwd=SEEDVC, env=env)

rep = json.loads((BENCH_DIR / "bench_stream.json").read_text(encoding="utf-8"))
b = rep["streaming"].get("balanced", {})
if b:
    print(f"""
خلاصة «الاستنساخ اللحظي» على {rep['gpu']}:
  • زمن معالجة كتلة البث (p95): {b['block_time_p95_ms']:.0f}ms — أي أن الصوت يخرج أسرع من دخوله بمعامل {b['rtf']:.2f}x
  • زمن الوصول الشامل المقدَّر: {b['estimated_e2e_ms']:.0f}ms (المرجع المنشور للنموذج نفسه: ~430ms)
  • RTF على مستوى المقطع (10 خطوات): {rep['utterance_rtf'].get('10', {}).get('rtf', '?')}
هذه أرقام جهازك الفعلي بالنموذج المضبوط — احتفظ بها (ستُرفع تلقائياً مع حزمة النموذج).""")

# استمع لناتج البث اللحظي الحقيقي (بروفايل balanced — نفس مسار الإنتاج)
from IPython.display import Audio, display
streamed = BENCH_DIR / "stream_balanced_out.wav"
if streamed.exists():
    print("\nاستمع لناتج البث اللحظي (كتل 180ms متلاحقة عبر محرك الإنتاج — بالنقطة المضبوطة):")
    display(Audio(filename=str(streamed)))


In [ ]:
# ============================================================ #
#  الخلية ٢٠ — تصدير الحزمة النهائية                           #
#  أوزان + إعداد + تقارير تقييم + بصمات إعادة الإنتاج          #
#  في مجلد ثابت على القرص الدائم: exports/<اسم-التشغيل>/       #
# ============================================================ #
import shutil, json
from pathlib import Path

FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — أكمل التدريب أولاً"
EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

sh([VENV_PY, PIPE, "export",
    "--run-dir", RUN_DIR,
    "--config", CFG_TINY,
    "--data-dir", DATA_DIR,
    "--out", EXPORT_DIR])

# ---- توثيق إضافي يُنسخ مع الحزمة (قابلية إعادة الإنتاج جزء من الجودة) ----
extras = [
    RUN_DIR / "train_console.log",
    RUN_DIR / "loss_curve.png",
    RUN_DIR / "config_dit_mel_seed_uvit_xlsr_tiny.yml",
    DATA_DIR / "_stats.json",
    ROOT / "data" / "duration_hist.png",        # مدرّج البيانات (جديد)
    ROOT / "repro.json",                         # بصمات git (جديد)
    ROOT / "nb_config.json",                     # مقابض التشغيل (جديد)
]
for extra in extras:
    if extra.exists():
        shutil.copy2(extra, EXPORT_DIR / extra.name)
        print(f"  + {extra.name}")

# ---- تضمين تقارير التقييم والبنشمارك داخل الحزمة (جديد) ----
for sub, dst_name in ((RUN_DIR / "ab_eval", "ab_eval"),
                      (RUN_DIR / "bench", "bench_stream")):
    if sub.exists():
        (EXPORT_DIR / dst_name).mkdir(exist_ok=True)
        for f in sub.iterdir():
            if f.is_file():
                shutil.copy2(f, EXPORT_DIR / dst_name / f.name)
        print(f"  + {dst_name}/ ({len(list((EXPORT_DIR / dst_name).iterdir()))} ملفات)")

# ---- دمج كل التقارير في ملف training_report.json واحد غني ----
report_path = EXPORT_DIR / "training_report.json"
report = json.loads(report_path.read_text(encoding="utf-8")) if report_path.exists() else {}
if (ROOT / "repro.json").exists():
    report["reproducibility"] = json.loads((ROOT / "repro.json").read_text(encoding="utf-8"))
ab_json = EXPORT_DIR / "ab_eval" / "ab_eval.json"
if ab_json.exists():
    report["ab_evaluation"] = json.loads(ab_json.read_text(encoding="utf-8"))["summary"]
    report["ab_evaluation"]["note"] = ("CAMPPlus speaker similarity + XLSR content "
                                       "preservation, held-out Iraqi pairs, identical seeds")
bench_json = EXPORT_DIR / "bench_stream" / "bench_stream.json"
if bench_json.exists():
    report["latency_benchmark"] = json.loads(bench_json.read_text(encoding="utf-8"))
report["training_config"] = {k: CFG[k] for k in CFG if not k.startswith("_") and k != "HF_TOKEN"}
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("  + training_report.json (محدّث بالتقييم والبنشمارك والبصمات)")

total = 0
print(f"\nحزمة التصدير: {EXPORT_DIR}")
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        s = p.stat().st_size
        total += s
        rel = p.relative_to(EXPORT_DIR)
        print(f"  {str(rel):<46} {s/1e6:9.2f} MB")
print(f"  {'الإجمالي':<46} {total/1e6:9.2f} MB")


<div dir="rtl">

## الحفظ الدائم — ثلاث طبقات (حتى تنزّلها بعد سنة)

| الطبقة | أين تبقى | كم تدوم | كيف تنزّلها لاحقاً |
|---|---|---|---|
| 1. القرص الشبكي RunPod | `/workspace/iraqi_seedvc/exports/<run>/` | ما دام الـ Volume موجوداً (لا تنسَ ألا تحذفه) | scp من لوحة RunPod ← Connect ← SSH، أو متصفح ملفات Jupyter |
| 2. HuggingFace Hub | مستودع خاص في **حسابك أنت** | **للأبد** ومجاني | `hf download <حسابك>/rafidain-seedvc-iraqi --local-dir ./model` |
| 3. أرشيف tar.gz | ملف واحد على القرص الشبكي | كالطبقة 1 | تنزيل مباشر من متصفح الملفات |

والمحتوى نفسه قابل للرفع لأي سحابة عبر rclone (الخلية 23) — طبقة رابعة اختيارية.

**توصية عملية:** لا تعتمد على طبقة واحدة. الحد الأدنى الموصى به: الطبقة 1 + الطبقة 2. الرفع إلى HuggingFace يستغرق دقائق ويجعل النموذج محصناً ضد أي شيء يحدث لحسابك في RunPod.

</div>


In [ ]:
# ============================================================ #
#  الخلية ٢١ — أرشيف tar.gz واحد جاهز للتنزيل                  #
# ============================================================ #
import shutil
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
assert (EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth").exists(), "شغّل خلية التصدير (20) أولاً"

base = EXPORTS / f"rafidain_seedvc_{CFG['RUN_NAME']}"
archive = shutil.make_archive(str(base), "gztar", root_dir=str(EXPORTS), base_dir=EXPORT_DIR.name)
size_mb = Path(archive).stat().st_size / 1e6
print(f"الأرشيف جاهز: {archive}  ({size_mb:.1f} MB)")
print(f'''
لتنزيله الآن:
  • من Jupyter: لوحة الملفات (يساراً) → مجلد exports → كليك يمين على الملف → Download
  • أو من جهازك عبر SSH (تجد الأمر في لوحة RunPod → Connect → SSH over TLS):
      scp -P <المنفذ> root@<العنوان>:{archive} .
''')


In [ ]:
# ============================================================ #
#  الخلية ٢٢ — الرفع الدائم إلى HuggingFace Hub                #
#  النموذج يُخزَّن في حسابك أنت، مجاناً، بلا علاقة بـ RunPod     #
#  (أعد تشغيلها في أي وقت مستقبلاً — ولو بعد شهور)              #
# ============================================================ #
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
HF_SCRIPT = ROOT / "hf_upload.py"
HF_SCRIPT.write_text(r'''
import os, time
from huggingface_hub import HfApi

token = os.environ["HF_TOKEN"]
export_dir = os.environ["NB_EXPORT_DIR"]
api = HfApi(token=token)
user = api.whoami()["name"]
repo_id = os.environ.get("NB_REPO_ID", "").strip() or f"{user}/rafidain-seedvc-iraqi"
private = os.environ.get("NB_PRIVATE", "1") == "1"
api.create_repo(repo_id, private=private, exist_ok=True)
print(f"[hf] رفع {export_dir} إلى {repo_id} (خاص = {private}) ...")
api.upload_folder(
    folder_path=export_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message=f"Rafidain Iraqi fine-tune ({time.strftime('%Y-%m-%d %H:%M')})",
)
print("[hf] REPO_URL: https://huggingface.co/" + repo_id)
''', encoding="utf-8")

if not CFG["HF_TOKEN"]:
    print("لا يوجد رمز HuggingFace بعد — لا مشكلة: كل أصولك محفوظة على القرص الشبكي،")
    print("ويمكنك تنفيذ الرفع الدائم في أي وقت لاحق (الخلية تعمل مستقلة):\n")
    print("  1) أنشئ حساباً مجانياً في huggingface.co إن لم يكن لديك")
    print("  2) من hf.co/settings/tokens أنشئ New Token بصلاحية Write وانسخه")
    print('  3) في الخلية 1 اضبط:  CFG["HF_TOKEN"] = "hf_ضع-الرمز-هنا"  ثم أعد تشغيلها')
    print("  4) أعد تشغيل هذه الخلية — سترفع الحزمة كاملة إلى مستودع خاص في حسابك")
else:
    assert (EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth").exists(), "شغّل خلية التصدير (20) أولاً"
    env = dict(ENV)
    env["HF_TOKEN"] = str(CFG["HF_TOKEN"])
    env["NB_EXPORT_DIR"] = str(EXPORT_DIR)
    env["NB_REPO_ID"] = str(CFG.get("HF_REPO_ID", ""))
    env["NB_PRIVATE"] = "1" if CFG["HF_PRIVATE"] else "0"
    sh([VENV_PY, HF_SCRIPT], env=env)
    print('''
[تم] النموذج الآن في حسابك على HuggingFace — بشكل دائم.
لتنزيله في أي وقت مستقبلاً من أي جهاز (بلا RunPod):
    pip install -U "huggingface_hub[cli]"
    hf download <اسم-حسابك>/rafidain-seedvc-iraqi --local-dir ./mymodel
أو من بايثون:
    from huggingface_hub import snapshot_download
    snapshot_download("<اسم-حسابك>/rafidain-seedvc-iraqi", local_dir="./mymodel")
أو مباشرة من صفحة المستودع في المتصفح.''')


In [ ]:
# ============================================================ #
#  الخلية ٢٣ — (اختيارية) طبقة رابعة: rclone سحابي             #
#  Google Drive / S3 / Dropbox / OneDrive ...                  #
#  خلية مرجعية — نفّذ الأوامر يدوياً وقت الحاجة                 #
# ============================================================ #

# 1) التثبيت (مرة واحدة على الحاوية):
#      apt-get install -y rclone          # أو أحدث إصدار من rclone.org/install
#
# 2) الإعداد التفاعلي (مرة واحدة — يربط حسابك السحابي):
#      rclone config                      # أنشئ remote باسم مثلاً: mydrive
#      (ملف الإعداد يبقى في /root/.config/rclone/rclone.conf — انسخه إلى
#       /workspace/iraqi_seedvc/ ليبقى مع القرص الدائم بين الحاويات)
#
# 3) رفع حزمة النموذج كاملة (أوزان + تقارير تقييم + بنشمارك):
#      rclone copy /workspace/iraqi_seedvc/exports/{run} mydrive:SeedVC_Iraqi/ -P
#
# 4) تنزيله لاحقاً من أي جهاز آخر فيه rclone مضبوط على الحساب نفسه:
#      rclone copy mydrive:SeedVC_Iraqi ./mymodel -P

print("خلية مرجعية — الأوامر جاهزة في التعليقات أعلاه.")
print("الطبقة الدائمة الأساسية تبقى HuggingFace Hub (الخلية 22) — الأسهل والأضمن.")


In [ ]:
# ============================================================ #
#  الخلية ٢٤ — (اختيارية) تحرير مساحة القرص                    #
#  تحذف المصادر الخام القابلة لإعادة الجلب فقط:                #
#    raw/yodas_webm + مخابئ parquet — وتبقى بيانات التدريب     #
#    المجهزة وكل النقاط والحزم كما هي                          #
#  ملاحظة: علامات .done تبقى، فلن يُعاد التنزيل تلقائياً؛       #
#  لإعادة الجلب مستقبلاً احذف ملف .done المناسب من raw/.done/  #
# ============================================================ #
import shutil
from pathlib import Path

UV = globals().get("UV") or shutil.which("uv")

def dir_gb(p: Path) -> float:
    if not p.exists():
        return 0.0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

targets = [ROOT / "raw" / "yodas_webm", ROOT / "raw" / "_parquet", ROOT / "raw" / "_safaa_src"]
print(f"المساحة الحرة الآن: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
for t in targets:
    s = dir_gb(t)
    print(f"  {t.name:<12} {s:6.2f} GB" + ("   (قابل لإعادة الجلب بالخليتين 7 و8)" if s > 0 else "   (فارغ)"))

if CFG["CLEAN_RAW_AFTER"]:
    for t in targets:
        if t.exists():
            shutil.rmtree(t)
            print(f"  حُذف: {t.name}")
    if UV:
        sh([UV, "cache", "clean"], check=False)
    else:
        sh([str(VENV_PY), "-m", "pip", "cache", "purge"], check=False)
    print(f"المساحة الحرة بعد التنظيف: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
    print("بيانات التدريب الجاهزة في data/iraqi_finetune — لا حاجة للمصادر الخام إلا لإعادة التجهيز.")
else:
    print('\nللتفعيل: اجعل  CFG["CLEAN_RAW_AFTER"] = True  في الخلية 1 ثم أعد تشغيل هذه الخلية.')


In [ ]:
# ============================================================ #
#  الخلية ٢٥ — النشر: كيف تستخدم النموذج المضبوط؟              #
#  (خلية إرشادية تطبع الأوامر الجاهزة بمساراتك الفعلية)        #
# ============================================================ #
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
W = EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth"
C = EXPORT_DIR / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

if not W.exists():
    print("شغّل خلية التصدير (20) أولاً حتى تتوفر الحزمة.")
else:
    # لوحة نتائج التقييم إن وُجدت (تذكير بأرقام هذا التشغيل)
    import json
    ab = EXPORT_DIR / "ab_eval" / "ab_eval.json"
    bench = EXPORT_DIR / "bench_stream" / "bench_stream.json"
    if ab.exists():
        s = json.loads(ab.read_text(encoding="utf-8"))["summary"]
        print(f"أرقام هذا التشغيل: تشابه متحدث {s['base_speaker_sim_mean']:.4f} ← {s['ft_speaker_sim_mean']:.4f} "
              f"(فرق {s['speaker_sim_delta']:+.4f}) | حفظ محتوى {s['ft_content_sim_mean']:.4f}")
    if bench.exists():
        b = json.loads(bench.read_text(encoding="utf-8"))["streaming"].get("balanced", {})
        if b:
            print(f"                   زمن وصول شامل (balanced): {b['estimated_e2e_ms']:.0f}ms | RTF {b['rtf']:.2f}")
    print(f'''
==========================================================
 نشر النموذج المضبوط — ثلاثة مسارات جاهزة
==========================================================

(أ) استدلال فوري في هذه الحاوية (البيئة جاهزة الآن):
    cd {SEEDVC}
    {VENV_PY} inference.py \\
        --checkpoint {W} \\
        --config     {C} \\
        --source <صوت-المتكلم-الأصلي.wav> \\
        --target  <صوت-مرجعي-عراقي.wav> \\
        --output  ./out

(ب) واجهة ويب Gradio للتجربة السريعة (تحتاج تثبيت gradio أولاً):
    uv pip install --python {VENV_PY} gradio   # أو: {VENV_PY} -m pip install gradio
    cd {SEEDVC}
    {VENV_PY} app_vc.py --checkpoint {W} --config {C}
    ثم افتح المنفذ 7860 (فعّل بروكسي HTTP من إعدادات الحاوية في لوحة RunPod)

(ج) خدمة رافِدَين الكاملة (REST + WebSocket لبث لحظي في المكالمات):
    المستودع منسوخ هنا: {RAFIDAIN}
    انسخ ما يلي في ملف .env الخاص بالخدمة (انظر .env.example داخل المستودع):

      RAFIDAIN__ENGINE__LOCAL_CHECKPOINT={W}
      RAFIDAIN__ENGINE__LOCAL_CONFIG={C}
      RAFIDAIN__ENGINE__DEVICE=cuda
      RAFIDAIN__API_KEYS=["<مفتاح-API-عشوائي-خاص-بك>"]

    التفاصيل الكاملة (Docker/GPU، الأمان، TLS، القياس):
      {RAFIDAIN}/DEPLOYMENT.md   و   {RAFIDAIN}/WEBUI.md

تذكير أخلاقي: استنساخ صوت شخص حقيقي يتطلب موافقته الصريحة الموثقة،
وأفصح للطرف الآخر عند ما يوجبه القانون — راجع قسم الأمان في README المستودع.
''')


<div dir="rtl">

## استكشاف الأخطاء

| المشكلة | الحل |
|---|---|
| `CUDA out of memory` أثناء التدريب | خفّض الدفعة: `CFG["BATCH"] = 2` (أو 1) في الخلية 1 ثم أعد تشغيلها والخلية 14 |
| الخسارة صارت NaN في المراقبة | أوقف التدريب (الخلية 14 تعيد الإطلاق من آخر نقطة سليمة تلقائياً)؛ إن تكرر: خفّض BATCH للنصف |
| توقفت عملية التدريب فجأة (خلع Pod / OOM نظام) | لا تفعل شيئاً غير إعادة تشغيل الخلية 14 — يستأنف من آخر `DiT_epoch_*_step_*.pth` |
| فشل تنزيل YODAS لبعض المصادر | أعد الخلية 8 — تعيد محاولة الفاشل فقط. عدد قليل من الفشل لا يؤثر (الكتالوج فيه 2360 مصدراً) |
| بطء تنزيل HuggingFace | اضبط `CFG["HF_ENDPOINT"] = "https://hf-mirror.com"` في الخلية 1 (مرآة) — أو العكس احذفها إن كانت مضبوطة |
| نفاد مساحة القرص | فعّل `CLEAN_RAW_AFTER` (الخلية 24) أو خفّض `YODAS_HOURS` أو وسّع الـ Volume من لوحة RunPod |
| التقييم A/B يقول NEUTRAL | النموذج لم يتعلم بعد: زد `STEPS` أو زد البيانات، ثم درّب فوق النقطة الحالية (استئناف) وأعد التقييم |
| النتيجة جيدة بالأرقام لكن الصوت «معدني» | هذا عادة الفوكودر لا الـ DiT — ارفع خطوات التدفق عند الاستدلال (`--diffusion-steps 30`) أو فكّر بضبط HiFT (متقدم) |
| التشابه انخفض بعد التدريب | جرعة زائدة (نسيان): درّب بعدد خطوات أقل، أو استخدم نقطة حفظ وسطية من `runs/<run>/` بدل النهائية |
| `verify` أنتج الصوت لكن A/B فشل | راجع رسالة السكربت — غالباً مقاطع اختبار أقل من المطلوب: خفّض `EVAL_PAIRS` أو شغّل خلية العزل 12 |
| أعدت تشغيل النواة وضاعت المتغيرات | طبيعي تماماً — شغّل الخلايا بالترتيب من 1: كل خلية تكتشف المنجز وتتخطاه |

## الخلاصة — أين كل شيء؟ (خريطة المقتنيات الدائمة)

| الأصل | المسار الدائم (على القرص الشبكي) |
|---|---|
| حزمة النموذج النهائية (أوزان + إعداد + تقارير) | `exports/<اسم-التشغيل>/` |
| الأرشيف المضغوط الواحد | `exports/rafidain_seedvc_<اسم-التشغيل>.tar.gz` |
| نسخة HuggingFace (للأبد) | `hf.co/<حسابك>/rafidain-seedvc-iraqi` |
| نقاط التدريب والسجلات | `seed-vc/runs/<اسم-التشغيل>/` |
| منحنى الخسارة + مدرّج البيانات | داخل حزمة التصدير |
| تقرير A/B + مقاطع المقارنة الصوتية | `exports/<run>/ab_eval/` |
| تقرير البنشمارك اللحظي | `exports/<run>/bench_stream/` |
| بيانات التدريب المجهزة (لجلسات قادمة) | `data/iraqi_finetune/` |
| بيانات الاختبار المعزولة | `data/iraqi_test/` |
| كل إعدادات التشغيل + بصمات git | `nb_config.json` + `repro.json` (وداخل الحزمة) |

**آخر خطوة دائماً:** شغّل الخلية 22 (HuggingFace) — دقيقتان تشتريانك راحة بال دائمة.

</div>
